In [ ]:
#@title **Protein-Ligand MD Pipeline and Analysis <元セル保持・GUI統合版>**
#@markdown Protein_ligand(2).ipynb の各コードセルを、元の処理内容をなるべく崩さずに統合したセルです。
#@markdown 実行する工程は RUN_MODE または CUSTOM_RUN_... で選びます。

# ============================================================
# 0. 共通import
# ============================================================

import os
import sys
import json
import re
import html as html_lib
import locale
import subprocess
import warnings
from pathlib import Path
from collections import OrderedDict
from statistics import mean, stdev

warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb

import pytraj as pt
from pytraj import matrix

import mdtraj as md
import parmed
import parmed as pmd

import MDAnalysis as mda
from MDAnalysis.analysis import align, rms
from MDAnalysis.lib.distances import distance_array

import py3Dmol
import prolif as plf
from prolif.plotting.network import LigNetwork

import openmm as mm
from openmm import *
from openmm.app import *
from openmm.unit import *
from openmm import app, unit
from openmm.app import HBonds, NoCutoff, PDBFile

import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit.Chem import rdMolTransforms
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem import rdDepictor
from rdkit.Chem import rdForceFieldHelpers
from rdkit.Chem.Draw import IPythonConsole

AllChem.SetPreferCoordGen(True)

from IPython.display import SVG, Image, display
import ipywidgets as widgets

from pdbfixer import PDBFixer
from openbabel import pybel
from biopandas.pdb import PandasPdb

import scipy.cluster.hierarchy
from scipy.spatial.distance import squareform
import scipy.stats as stats
from scipy.interpolate import griddata
from matplotlib import colors

def getpreferredencoding(do_setlocale=True):
    return "UTF-8"

locale.getpreferredencoding = getpreferredencoding

print("Imports: OK")


# ============================================================
# 1. 実行する工程の選択
# ============================================================
# Colab標準の #@param GUIのみを使う。
# 元セルのMD本体・topology・平衡化・production・解析処理は、各ブロック内で元セルのまま実行する。
#
# RUN_MODE:
#   all              → 表示系・デバッグ系以外の主要工程を一気に実行
#   all_with_visuals → 主要工程 + 3D表示系も実行
#   analysis_only    → 既存trajectoryを使って解析だけ実行
#   debug_only       → デバッグ系だけ実行
#   custom           → 下の個別チェックボックスを使う
#   off              → 全部OFF
# ============================================================

RUN_MODE = "custom" #@param ["all", "all_with_visuals", "analysis_only", "debug_only", "custom", "off"]

CUSTOM_RUN_SYSTEM_CHECK = False #@param {type:"boolean"}  # 0.5 GPU/Python/working directory check
CUSTOM_RUN_SET_WORKDIR = True #@param {type:"boolean"}  # 1. Set work directory
CUSTOM_RUN_PREPARE_INPUTS = False #@param {type:"boolean"}  # 2. Prepare protein and ligand input files
CUSTOM_RUN_GAP_DEBUG = False #@param {type:"boolean"}  # 2.5 Extract chain-gap debug PDB
CUSTOM_RUN_INSERT_TER = False #@param {type:"boolean"}  # 3. Insert TER after chain breaks
CUSTOM_RUN_CHECK_NA_POSITION = False #@param {type:"boolean"}  # 3.5 Check NA position relative to ligand
CUSTOM_RUN_TOPOLOGY = False #@param {type:"boolean"}  # 4. Generate topology
CUSTOM_RUN_SHOW_INITIAL_STRUCTURE = False #@param {type:"boolean"}  # 5. Show initial 3D structure
CUSTOM_RUN_INITIAL_LIGPLOT = False #@param {type:"boolean"}  # 6. Initial LigPlot
CUSTOM_RUN_EQUILIBRATION = False #@param {type:"boolean"}  # 7. Equilibration MD
CUSTOM_RUN_PRODUCTION = False #@param {type:"boolean"}  # 8. Production MD
CUSTOM_RUN_CONCAT_TRAJECTORY = False #@param {type:"boolean"}  # 9. Concatenate and align trajectory
CUSTOM_RUN_VIEW_TRAJECTORY = False #@param {type:"boolean"}  # 10. Load, view and check trajectory
CUSTOM_RUN_MD_LIGPLOT = False #@param {type:"boolean"}  # 11. MD LigPlot
CUSTOM_RUN_MMPBSA = False #@param {type:"boolean"}  # 12. MM-PBSA / MM-GBSA
CUSTOM_RUN_INTERACTION_ENERGY_DEBUG = False #@param {type:"boolean"}  # 12.5 Interaction energy debug
CUSTOM_RUN_INTERACTION_ENERGY = False #@param {type:"boolean"}  # 13. Interaction Energy
CUSTOM_RUN_DISTANCE_NEAREST_RESIDUES = False #@param {type:"boolean"}  # 14. Distance to nearest residues
CUSTOM_RUN_DISTANCE_SPECIFIC_RESIDUES_MIN = False #@param {type:"boolean"}  # 15. Minimum distance to selected residues
CUSTOM_RUN_DETECT_NEARBY_RESIDUES = False #@param {type:"boolean"}  # 15.5 Detect nearby residues
CUSTOM_RUN_RMSD = False #@param {type:"boolean"}  # 16. RMSD
CUSTOM_RUN_RMSD_DISTRIBUTION = False #@param {type:"boolean"}  # 17. RMSD distribution
CUSTOM_RUN_RADIUS_GYRATION = False #@param {type:"boolean"}  # 18. Radius of gyration
CUSTOM_RUN_RADIUS_GYRATION_DISTRIBUTION = False #@param {type:"boolean"}  # 19. Radius of gyration distribution
CUSTOM_RUN_RMSF = False #@param {type:"boolean"}  # 20. RMSF
CUSTOM_RUN_2D_RMSD = False #@param {type:"boolean"}  # 21. 2D RMSD
CUSTOM_RUN_PCA = False #@param {type:"boolean"}  # 22. PCA
CUSTOM_RUN_PCA_DISTRIBUTION = False #@param {type:"boolean"}  # 23. PCA distribution
CUSTOM_RUN_CROSS_CORRELATION = False #@param {type:"boolean"}  # 24. Pearson Cross Correlation
CUSTOM_RUN_CONTACT_FREQUENCY_RANGE = False #@param {type:"boolean"}  # 25. Contact frequency for selected residue range


ALL_RUN_FLAGS = [
    "RUN_SYSTEM_CHECK",
    "RUN_SET_WORKDIR",
    "RUN_PREPARE_INPUTS",
    "RUN_GAP_DEBUG",
    "RUN_INSERT_TER",
    "RUN_CHECK_NA_POSITION",
    "RUN_TOPOLOGY",
    "RUN_SHOW_INITIAL_STRUCTURE",
    "RUN_INITIAL_LIGPLOT",
    "RUN_EQUILIBRATION",
    "RUN_PRODUCTION",
    "RUN_CONCAT_TRAJECTORY",
    "RUN_VIEW_TRAJECTORY",
    "RUN_MD_LIGPLOT",
    "RUN_MMPBSA",
    "RUN_INTERACTION_ENERGY_DEBUG",
    "RUN_INTERACTION_ENERGY",
    "RUN_DISTANCE_NEAREST_RESIDUES",
    "RUN_DISTANCE_SPECIFIC_RESIDUES_MIN",
    "RUN_DETECT_NEARBY_RESIDUES",
    "RUN_RMSD",
    "RUN_RMSD_DISTRIBUTION",
    "RUN_RADIUS_GYRATION",
    "RUN_RADIUS_GYRATION_DISTRIBUTION",
    "RUN_RMSF",
    "RUN_2D_RMSD",
    "RUN_PCA",
    "RUN_PCA_DISTRIBUTION",
    "RUN_CROSS_CORRELATION",
    "RUN_CONTACT_FREQUENCY_RANGE",
]

DEBUG_RUN_FLAGS = {
    "RUN_CHECK_NA_POSITION",
    "RUN_DETECT_NEARBY_RESIDUES",
    "RUN_GAP_DEBUG",
    "RUN_INTERACTION_ENERGY_DEBUG",
    "RUN_SYSTEM_CHECK",
}

VISUAL_RUN_FLAGS = {
    "RUN_INITIAL_LIGPLOT",
    "RUN_SHOW_INITIAL_STRUCTURE",
    "RUN_VIEW_TRAJECTORY",
}

MAJOR_RUN_FLAGS = {
    "RUN_SET_WORKDIR",
    "RUN_PREPARE_INPUTS",
    "RUN_INSERT_TER",
    "RUN_TOPOLOGY",
    "RUN_EQUILIBRATION",
    "RUN_PRODUCTION",
    "RUN_CONCAT_TRAJECTORY",
    "RUN_MD_LIGPLOT",
    "RUN_MMPBSA",
    "RUN_INTERACTION_ENERGY",
    "RUN_DISTANCE_NEAREST_RESIDUES",
    "RUN_DISTANCE_SPECIFIC_RESIDUES_MIN",
    "RUN_RMSD",
    "RUN_RMSD_DISTRIBUTION",
    "RUN_RADIUS_GYRATION",
    "RUN_RADIUS_GYRATION_DISTRIBUTION",
    "RUN_RMSF",
    "RUN_2D_RMSD",
    "RUN_PCA",
    "RUN_PCA_DISTRIBUTION",
    "RUN_CROSS_CORRELATION",
    "RUN_CONTACT_FREQUENCY_RANGE",
}

ANALYSIS_RUN_FLAGS = {
    "RUN_CONCAT_TRAJECTORY",
    "RUN_MD_LIGPLOT",
    "RUN_MMPBSA",
    "RUN_INTERACTION_ENERGY",
    "RUN_DISTANCE_NEAREST_RESIDUES",
    "RUN_DISTANCE_SPECIFIC_RESIDUES_MIN",
    "RUN_RMSD",
    "RUN_RMSD_DISTRIBUTION",
    "RUN_RADIUS_GYRATION",
    "RUN_RADIUS_GYRATION_DISTRIBUTION",
    "RUN_RMSF",
    "RUN_2D_RMSD",
    "RUN_PCA",
    "RUN_PCA_DISTRIBUTION",
    "RUN_CROSS_CORRELATION",
    "RUN_CONTACT_FREQUENCY_RANGE",
}


def _set_all_run_flags(value=False):
    for _name in ALL_RUN_FLAGS:
        globals()[_name] = bool(value)

_set_all_run_flags(False)

if RUN_MODE == "off":
    pass

elif RUN_MODE == "all":
    for _name in MAJOR_RUN_FLAGS:
        globals()[_name] = True

elif RUN_MODE == "all_with_visuals":
    for _name in (MAJOR_RUN_FLAGS | VISUAL_RUN_FLAGS):
        globals()[_name] = True

elif RUN_MODE == "analysis_only":
    for _name in ANALYSIS_RUN_FLAGS:
        globals()[_name] = True
    RUN_SET_WORKDIR = True

elif RUN_MODE == "debug_only":
    for _name in DEBUG_RUN_FLAGS:
        globals()[_name] = True
    RUN_SET_WORKDIR = True

elif RUN_MODE == "custom":
    for _name in ALL_RUN_FLAGS:
        globals()[_name] = globals().get("CUSTOM_" + _name, False)

else:
    raise ValueError("RUN_MODE が不正です。")

print("=== Effective RUN settings ===")
print("RUN_MODE:", RUN_MODE)
for _name in ALL_RUN_FLAGS:
    print(f"{_name:40s} = {globals().get(_name, False)}")


# ============================================================
# 0.5 GPU/Python/working directory check
# ============================================================

if RUN_SYSTEM_CHECK:
    print('\n=== 0.5 GPU/Python/working directory check ===')
    # --- Original notebook cell 6 ---
    # Colabでこれ実行 → 歯学部GPU情報出れば成功！
    !nvidia-smi

    # --- Original notebook cell 7 ---
    import sys
    print(sys.executable)

    import pytraj as pt
    print(pt.__version__)

    # --- Original notebook cell 13 ---
    #chatgpt修正版 2つのpdbファイルがちゃんと存在するか調べる
    !pwd
    !ls -la /home/gregorymaddux18/making_it_rain_work



# ============================================================
# 1. Set work directory
# ============================================================

if RUN_SET_WORKDIR:
    print('\n=== 1. Set work directory ===')
    # --- Original notebook cell 11 ---
    # chatgpt修正版 googleドライブにマウントしない代わりに作業ディレクトリをここに置く。今後のファイルは基本ここに置く
    import os

    project_dir = "/home/gregorymaddux18/making_it_rain_work"
    os.makedirs(project_dir, exist_ok=True)
    os.chdir(project_dir)

    print("現在の作業場所:")
    print(os.getcwd())



# ============================================================
# 2. Prepare protein and ligand input files
# ============================================================

if RUN_PREPARE_INPUTS:
    print('\n=== 2. Prepare protein and ligand input files ===')
    # --- Original notebook cell 12 ---
    #chatgpt修正版
    import os
    import sys
    import subprocess
    import warnings

    from biopandas.pdb import PandasPdb
    from openmm.app import PDBFile

    # --- Original notebook cell 15 ---
    #chatgpt修正版

    #@title **Please, provide the necessary input files below**:
    #@markdown **Important:** The protonation of your ligand is crucial for the correct parameterization of the molecule.
    # %%capture
    import rdkit
    import mdtraj as md
    from rdkit import Chem
    from rdkit.Chem import AllChem
    from rdkit.Chem import Draw
    from rdkit.Chem import rdMolTransforms
    from rdkit.Chem.Draw import rdMolDraw2D
    from rdkit.Chem import rdDepictor
    from rdkit.Chem import rdForceFieldHelpers
    from IPython.display import SVG
    import ipywidgets as widgets
    import rdkit
    from rdkit.Chem.Draw import IPythonConsole
    AllChem.SetPreferCoordGen(True)
    from IPython.display import Image
    from pdbfixer import PDBFixer
    from openbabel import pybel
    import os
    import subprocess
    import warnings

    # Suppress UserWarnings
    warnings.filterwarnings("ignore", category=UserWarning)


    Protein_PDB_file_name = '7M8W.pdb' #@param {type:"string"}
    remove_waters = "no" #@param ["yes", "no" ]
    if remove_waters == "yes":
      no_waters = "nowat"
    else:
      no_waters = ''

    Ligand_PDB_file_name = 'PGD2.pdb'  #@param {type:"string"}

    Add_ligand_hydrogens = "No" #@param ["Yes", "No"]
    Charge = -1 #@param {type:"slider", min:-10, max:10, step:1}


    ligand_name = Ligand_PDB_file_name
    # Google_Drive_Path = '/content/drive/MyDrive/' #@param {type:"string"}
    workDir = project_dir
    os.chdir(workDir)
    initial_pdb = os.path.join(workDir, str(Protein_PDB_file_name))
    prepareforleap = os.path.join(workDir, "prepareforleap.in")
    ligand_pdb = os.path.join(workDir, str(ligand_name))
    ligand_pdb2 = os.path.join(workDir, "ligand_H.pdb")
    starting = os.path.join(workDir, "starting1.pdb")
    starting2 = os.path.join(workDir, "starting2.pdb")
    starting_end = os.path.join(workDir, "starting_end.pdb")


    def mol_with_atom_index(mol):
        for atom in mol.GetAtoms():
            atom.SetAtomMapNum(atom.GetIdx())
        return mol

    def remove_lines(filename):
        with open(filename, 'r') as file:
            ter_count = 0
            for line in file:
                if line.startswith('TER'):
                    ter_count += 1
                    if ter_count >= 1:
                        yield line
                        for i in range(3):
                            line = next(file, None)
                            if line is not None and line.startswith('ATOM') and line.split()[2] in ['P', 'OP1', 'OP2']:
                                continue
                            else:
                                yield line
                    else:
                        yield line
                else:
                    yield line

    # if Add_ligand_hydrogens == "Yes":
    #   mol= [m for m in pybel.readfile(filename=ligand_pdb, format='pdb')][0]
    #   out=pybel.Outputfile(filename="temp.mol",format='mol',overwrite=True)
    #   out.write(mol)
    #   out.close()

    #   mol = Chem.MolFromMolFile('temp.mol', removeHs=True)
    #   hmol = Chem.AddHs(mol)
    #   mp = AllChem.MMFFGetMoleculeProperties(hmol)
    #   ff = AllChem.MMFFGetMoleculeForceField(hmol, mp)
    #   for a in hmol.GetAtoms():
    #     if (a.GetAtomicNum() > 1):
    #       ff.MMFFAddPositionConstraint(a.GetIdx(), 0, 1.e4)
    #   ff.Minimize(maxIts=1000)
    #   charge_mol = Chem.rdPartialCharges.ComputeGasteigerCharges(hmol)
    #   charge = Chem.GetFormalCharge(hmol)
    #   print("Charge = " + str(charge))
    #   # AllChem.MolToMolFile(hmol, (os.path.join(workDir, f"start_min.mol")))
    #   AllChem.MolToPDBFile(hmol, ligand_pdb2)
    #   mol_end = mol_with_atom_index(hmol)
    #   IPythonConsole.drawMol3D(hmol)
    # else:
    #   mol= [m for m in pybel.readfile(filename=ligand_pdb, format='pdb')][0]
    #   out=pybel.Outputfile(filename="temp.mol",format='mol',overwrite=True)
    #   out.write(mol)
    #   out.close()

    #   hmol = Chem.MolFromMolFile('temp.mol', removeHs=False)
    #   mp = AllChem.MMFFGetMoleculeProperties(hmol)
    #   ff = AllChem.MMFFGetMoleculeForceField(hmol, mp)
    #   for a in hmol.GetAtoms():
    #     if (a.GetAtomicNum() > 1):
    #       ff.MMFFAddPositionConstraint(a.GetIdx(), 0, 1.e4)
    #   ff.Minimize(maxIts=1000)
    #   charge_mol = Chem.rdPartialCharges.ComputeGasteigerCharges(hmol)
    #   charge = Chem.GetFormalCharge(hmol)
    #   print("Charge = " + str(charge))
    #   # AllChem.MolToMolFile(hmol, (os.path.join(workDir, f"start_min.mol")))
    #   AllChem.MolToPDBFile(hmol, ligand_pdb2)
    #   mol_end = mol_with_atom_index(hmol)
    #   IPythonConsole.drawMol3D(hmol)



    #Add hydrogens in the ligand
    if Add_ligand_hydrogens == "Yes":
      fixer = PDBFixer(filename=ligand_pdb)
      PDBFile.writeFile(fixer.topology, fixer.positions, open("temp.pdb", 'w'))

      ppdb = PandasPdb().read_pdb("temp.pdb")
      ppdb.df['ATOM'] = ppdb.df['ATOM']
      ppdb.df['HETATM']= ppdb.df['HETATM'][ppdb.df['HETATM']['element_symbol'] != 'H']
      ppdb.to_pdb(path="temp.pdb", records=['ATOM', 'HETATM'], gz=False, append_newline=True)

      mol= [m for m in pybel.readfile(filename="temp.pdb", format='pdb')][0]
      mol.calccharges
      mol.addh()
      out=pybel.Outputfile(filename="temp2.pdb",format='pdb',overwrite=True)
      out.write(mol)
      out.close()

      md.load("temp2.pdb").save("temp2.pdb")

      halogens = ['Cl', 'F', 'Br', 'I']
      atom_id = []
      H_id = []
      with open("temp2.pdb") as f:
          for line in f:
            data = line.split()
            if data[0] == "ATOM":
              if data[2] in halogens:
                atom_id.append(data[1])
            if data[0] == "CONECT":
              if data[1] in atom_id:
                if len(data) > 3:
                  H_id.append(data[3])
                  H_id.append(data[4])
                  H_id.append(data[5])

      # with open(ligand_pdb2, 'w') as h:
      #   with open("temp2.pdb") as f:
      #     for line in f:
      #       data = line.split()
      #       if data[0] == "ATOM":
      #         if data[1] not in H_id:
      #           print(line, file=h)
      #       elif data[0] == "CONECT":
      #           if data[1] not in atom_id:
      #             print(line, file=h)
      #       else:
      #         print(line, file=h)

      with open(ligand_pdb2, 'w') as h:
        with open("temp2.pdb") as f:
            for line in f:
                if line.strip():  # Check if line is not empty or just whitespace
                    data = line.split()
                    if len(data) > 0 and data[0] not in ["TER", "ENDMDL"]:  # Exclude lines starting with TER or ENDMDL
                        if data[0] == "ATOM":
                            if data[1] not in H_id:
                                print(line, end='', file=h)  # Avoid adding extra newline
                        elif data[0] == "CONECT":
                            if data[1] not in atom_id:
                                print(line, end='', file=h)
                        else:
                            print(line, end='', file=h)

      mol= [m for m in pybel.readfile(filename=ligand_pdb2, format='pdb')][0]
      out=pybel.Outputfile(filename="temp.mol",format='mol',overwrite=True)
      out.write(mol)
      out.close()
      hmol = Chem.MolFromMolFile('temp.mol', removeHs=False)
      # charge_mol = Chem.rdPartialCharges.ComputeGasteigerCharges(hmol)
      charge = Charge
      print("Charge = " + str(charge))
      mol_end = mol_with_atom_index(hmol)
      IPythonConsole.drawMol3D(hmol)

    else:
      ppdb = PandasPdb().read_pdb(ligand_pdb)
      ppdb.df['ATOM'] = ppdb.df['ATOM']
      ppdb.to_pdb(path="temp.pdb", records=['ATOM', 'HETATM'], gz=False, append_newline=True)
      mol= [m for m in pybel.readfile(filename="temp.pdb", format='pdb')][0]
      mol.calccharges
      out=pybel.Outputfile(filename="temp2.pdb",format='pdb',overwrite=True)
      out.write(mol)
      out.close()

      md.load("temp2.pdb").save("temp2.pdb")

      with open(ligand_pdb2, 'w') as h:
        with open("temp2.pdb") as f:
            for line in f:
                if line.strip() and not line.startswith(("TER", "ENDMDL")):
                    print(line, end='', file=h)

      mol= [m for m in pybel.readfile(filename=ligand_pdb2, format='pdb')][0]
      out=pybel.Outputfile(filename="temp.mol",format='mol',overwrite=True)
      out.write(mol)
      out.close()
      hmol = Chem.MolFromMolFile('temp.mol', removeHs=False)
      # charge_mol = Chem.rdPartialCharges.ComputeGasteigerCharges(hmol)
      charge = Charge
      print("Charge = " + str(charge))
      mol_end = mol_with_atom_index(hmol)
      IPythonConsole.drawMol3D(hmol)


    #Fix protein
    f = open(prepareforleap, "w")
    f.write("""parm """ + str(initial_pdb) + "\n"
    """loadcrd """ + str(initial_pdb) + """ name edited""" + "\n"
    """prepareforleap crdset edited name from-prepareforleap \ """ + "\n"
    """pdbout """ + str(starting) + " " + str(no_waters) + """ noh""" + "\n"
    """go """)
    f.close()

    prepareforleap_command = "cpptraj -i " + str(prepareforleap)
    original_stdout = sys.stdout # Save a reference to the original standard output
    with open('prepareforleap.sh', 'w') as f:
        sys.stdout = f # Change the standard output to the file we created.
        print(prepareforleap_command)
        sys.stdout = original_stdout # Reset the standard output to its original value

    subprocess.run(["chmod 700 prepareforleap.sh"], shell=True)
    subprocess.run(["./prepareforleap.sh"], shell=True,)


    pdb4amber_cmd = "pdb4amber -i " + str(starting) + " -o " + str(starting_end) + " -a"
    original_stdout = sys.stdout # Save a reference to the original standard output

    with open('pdb4amber.sh', 'w') as f:
        sys.stdout = f # Change the standard output to the file we created.
        print(pdb4amber_cmd)
        sys.stdout = original_stdout # Reset the standard output to its original value

    subprocess.run(["chmod 700 pdb4amber.sh"], shell=True)
    subprocess.run(["./pdb4amber.sh"], shell=True,)

    # with open(starting_end, 'w') as out_file:
    #     for line in remove_lines(starting2):
    #         out_file.write(line)

    protein_check = os.path.exists(starting_end)
    ligand_check = os.path.exists(ligand_pdb2)

    if protein_check == True and ligand_check == True:
      print("Successfully generated protein and ligand files! :-)")
    else:
      print("ERROR: Check your inputs! ")

    #@markdown ---



# ============================================================
# 2.5 Extract chain-gap debug PDB
# ============================================================

if RUN_GAP_DEBUG:
    print('\n=== 2.5 Extract chain-gap debug PDB ===')
    # --- Original notebook cell 16 ---
    #chatgpt修正版 2残基だけのPDBを作る
    gap_pdb = "gap_GLU246_247.pdb"

    with open("starting_end.pdb") as fin, open(gap_pdb, "w") as fout:
        for line in fin:
            if line.startswith(("ATOM", "HETATM")):
                resname = line[17:20].strip()
                chain = line[21].strip()
                resnum = line[22:26].strip()
                if resname == "GLU" and chain == "A" and resnum in {"246", "247"}:
                    fout.write(line)
        fout.write("END\n")

    print("created:", gap_pdb)

    # --- Original notebook cell 21 ---
    gap_pdb = "gap_GLU246_247.pdb"

    with open("starting_end.pdb") as fin, open(gap_pdb, "w") as fout:
        for line in fin:
            if line.startswith(("ATOM", "HETATM")):
                resname = line[17:20].strip()
                chain = line[21].strip()
                resnum = line[22:26].strip()
                if resname == "GLU" and chain == "A" and resnum in {"246", "247"}:
                    fout.write(line)
        fout.write("END\n")

    print("created:", gap_pdb)



# ============================================================
# 3. Insert TER after chain breaks
# ============================================================

if RUN_INSERT_TER:
    print('\n=== 3. Insert TER after chain breaks ===')
    # --- Original notebook cell 18 ---
    #chatgpt修正版
    # GLU246 と THR414 の後に TER を入れて、
    # ここでタンパク質鎖は切れてますと Amber に教える

    import os

    def insert_ter_after_residues(in_pdb, out_pdb, break_points):
        """
        break_points に指定した残基の直後へ TER を入れる。
        例: {("A", 246), ("A", 414)}
        """
        break_points = {(chain_id, str(residue_number)) for chain_id, residue_number in break_points}
        inserted = set()
        prev_key = None

        with open(in_pdb, "r") as fin, open(out_pdb, "w") as fout:
            for line in fin:
                if line.startswith(("ATOM", "HETATM")):
                    chain = line[21].strip()
                    resnum = line[22:26].strip()
                    current_key = (chain, resnum)

                    # 指定残基の次の残基に移った瞬間に TER を入れる
                    if prev_key in break_points and current_key != prev_key and prev_key not in inserted:
                        fout.write("TER\n")
                        inserted.add(prev_key)

                    prev_key = current_key

                fout.write(line)

            # ファイル末尾まで指定残基だった場合の保険
            if prev_key in break_points and prev_key not in inserted:
                fout.write("TER\n")

    starting_end_fixed = os.path.join(workDir, "starting_end_TER.pdb")

    insert_ter_after_residues(
        in_pdb=starting_end,
        out_pdb=starting_end_fixed,
        break_points={
            ("A", 246),  # GLU246 の後
            ("A", 414),  # THR414 の後
        }
    )

    print("created:", starting_end_fixed)



# ============================================================
# 3.5 Check NA position relative to ligand
# ============================================================

if RUN_CHECK_NA_POSITION:
    print('\n=== 3.5 Check NA position relative to ligand ===')
    # --- Original notebook cell 19 ---
    #chatgpt 修正版 NA(ナトリウムイオン)のリガンドからの相対位置を確認する

    import numpy as np

    def read_pdb_coords(pdb_file, target_resname=None, target_atomname=None):
        coords = []
        lines = []

        with open(pdb_file) as f:
            for line in f:
                if line.startswith(("ATOM", "HETATM")):
                    atomname = line[12:16].strip()
                    resname = line[17:20].strip()

                    if target_resname is not None and resname != target_resname:
                        continue
                    if target_atomname is not None and atomname != target_atomname:
                        continue

                    x = float(line[30:38])
                    y = float(line[38:46])
                    z = float(line[46:54])

                    coords.append([x, y, z])
                    lines.append(line.rstrip())

        return np.array(coords), lines

    def min_distance(coords1, coords2):
        if len(coords1) == 0 or len(coords2) == 0:
            return None
        diff = coords1[:, None, :] - coords2[None, :, :]
        dists = np.sqrt((diff * diff).sum(axis=2))
        return dists.min()

    na_coords, na_lines = read_pdb_coords("protein_ligand.pdb", target_resname="NA")
    lig_coords, lig_lines = read_pdb_coords("protein_ligand.pdb", target_resname="LIG")

    print("NA lines:")
    for line in na_lines:
        print(line)

    print("Number of NA atoms:", len(na_coords))
    print("Number of ligand atoms:", len(lig_coords))
    print("Minimum distance NA - LIG:", min_distance(na_coords, lig_coords), "Å")



# ============================================================
# 4. Generate topology
# ============================================================

if RUN_TOPOLOGY:
    print('\n=== 4. Generate topology ===')
    # --- Original notebook cell 20 ---
    #chrgpt修正版：topology生成セルのみ

    from rdkit import Chem
    import parmed

    #@title **Parameters to generate the topology:**

    #@markdown **Parameters to generate the protein topology:**

    Force_field = "ff19SB" #@param ["ff19SB", "ff14SB"]
    if Force_field == "ff19SB":
      ff = "leaprc.protein.ff19SB"
    else:
      ff = "leaprc.protein.ff14SB"

    Water_type = "OPC" #@param ["TIP3P", "OPC"]
    if Water_type == "TIP3P":
      water = "leaprc.water.tip3p"
      water_box = "TIP3PBOX"
    else:
      water = "leaprc.water.opc"
      water_box = "OPCBOX"

    #@markdown Size Box (Angstrons):

    Size_box = 12 #@param {type:"slider", min:10, max:20, step:1}
    size_box = Size_box

    #@markdown **ATTENTION**: Give the concentration in Molar units:

    Ions = "NaCl" #@param ["NaCl", "KCl" ]

    Concentration = "0.15" #@param {type:"string"}

    #@markdown **Parameters to generate the ligand topology:**

    Ligand_Force_field = "GAFF2" #@param ["GAFF2"]

    Charge = -1 #@param {type:"slider", min:-10, max:10, step:1}
    Ligand_net_charges = Charge

    #@markdown ---

    tleap = os.path.join(workDir, "tleap.in")
    top_nw = os.path.join(workDir, "SYS_nw.prmtop")
    crd_nw = os.path.join(workDir, "SYS_nw.crd")
    pdb_nw = os.path.join(workDir, "SYS_nw.pdb")
    top = os.path.join(workDir, "SYS_gaff2.prmtop")
    crd = os.path.join(workDir, "SYS_gaff2.crd")
    pdb = os.path.join(workDir, "SYS.pdb")
    ligand_noh = os.path.join(workDir, "ligand_noh.pdb")
    ligand_h = os.path.join(workDir, "ligand_h.pdb")
    ligand_mol2 = os.path.join(workDir, "ligand.mol2")
    ligand_frcmod = os.path.join(workDir, "ligand.frcmod")
    lig_new = os.path.join(workDir, "ligand_gaff.pdb")
    protein_ligand = os.path.join(workDir, "protein_ligand.pdb")
    lib = os.path.join(workDir, "lig.lib")

    top_tmp = os.path.join(workDir, "SYS_tmp.prmtop")
    crd_tmp = os.path.join(workDir, "SYS_tmp.crd")
    pdb_tmp = os.path.join(workDir, "SYS_tmp.pdb")

    # 古い出力を削除
    !rm -f SYS* ligand_h.pdb ligand.mol2 ligand.frcmod ligand_gaff.pdb lig.lib leap.log *.sh ANTECHAMBER* ATOMTYPE* temp.txt sqm.in sqm.out

    # ligand parameterization
    gaff_command1 = "pdb4amber -i " + str(ligand_pdb2) + " -o " + str(ligand_h)
    gaff_command3 = "antechamber -i " + str(ligand_h) + " -fi pdb -o " + str(ligand_mol2) + " -fo mol2 -c bcc -nc " + str(Ligand_net_charges) + " -rn LIG -at gaff2"
    gaff_command4 = "parmchk2 -i " + str(ligand_mol2) + " -f mol2 -o " + str(ligand_frcmod) + " -s gaff2"

    original_stdout = sys.stdout

    with open('gaff.sh', 'w') as f:
        sys.stdout = f
        print(gaff_command1)
        print(gaff_command3)
        print(gaff_command4)
        sys.stdout = original_stdout

    !chmod 700 gaff.sh
    !bash gaff.sh

    # ligand library作成
    f = open(tleap, "w")
    f.write("""source """ + str(ff) + "\n"
    """source leaprc.gaff2
    LIG = loadmol2 """ + str(ligand_mol2) + "\n"
    """loadamberparams """ + str(ligand_frcmod) + "\n"
    """saveoff LIG """ + str(lib) + "\n"
    """savepdb LIG """ + str(lig_new) + "\n"
    """quit""")
    f.close()

    # ligand library作成を実行
    tleap_command = "tleap -f " + str(tleap)

    original_stdout = sys.stdout

    with open('run_tleap.sh', 'w') as f:
        sys.stdout = f
        print(tleap_command)
        sys.stdout = original_stdout

    !chmod 700 run_tleap.sh
    !bash run_tleap.sh

    # protein + ligand PDB を安全に結合する
    # END / CONECT は除外し、ATOM/HETATM/TER だけを書き出す
    # 最後に END を1回だけ入れる
    with open(protein_ligand, "w") as out:
        for pdb_file in [starting_end_fixed, lig_new]:
            with open(pdb_file, "r") as f:
                for line in f:
                    if line.startswith(("ATOM", "HETATM", "TER")):
                        out.write(line)
        out.write("END\n")

    print("created:", protein_ligand)

    print("LIG check in protein_ligand.pdb:")
    !grep -n "LIG" protein_ligand.pdb | head

    # YCM, SO4, NAを削除
    target_pdb = protein_ligand

    if os.path.exists(target_pdb):
        with open(target_pdb, 'r') as f:
            lines = f.readlines()

        removed_count = 0

        with open(target_pdb, 'w') as f:
            for line in lines:
                if line.startswith(("ATOM", "HETATM")):
                    resname = line[17:20].strip()
                    atomname = line[12:16].strip()

                    if resname in {"YCM", "SO4", "NA"} or atomname == "NA":
                        removed_count += 1
                        continue

                f.write(line)

        print(f"完了！ {target_pdb} から YCM, SO4, NA を削除しました。")
        print(f"削除した行数: {removed_count}")
    else:
        print(f"エラー: {target_pdb} が見つかりません。パスを確認してください。")

    # 1回目tleap：イオンなしで水を入れてVolumeを取得
    f = open(tleap, "w")
    f.write("""source """ + str(ff) + "\n"
    """source leaprc.gaff2
    source """  + str(water) + "\n"
    """loadamberparams """ + str(ligand_frcmod) + "\n"
    """loadoff """ + str(lib) + "\n"
    """SYS = loadpdb """ + str(protein_ligand) + "\n"
    """alignaxes SYS
    savepdb SYS """ + str(pdb_nw) + "\n"
    """saveamberparm SYS """ + str(top_nw) + " " + str(crd_nw) + "\n"
    """solvatebox SYS """ + str(water_box) + " " + str(size_box) +  "  0.7\n"
    """charge SYS\n"""
    """saveamberparm SYS """ + str(top_tmp) + " " + str(crd_tmp) + "\n"
    """savepdb SYS """ + str(pdb_tmp) + "\n"
    """quit""")
    f.close()

    tleap_command = "tleap -f " + str(tleap)

    with open('run_tleap.sh', 'w') as f:
        print(tleap_command, file=f)

    !chmod 700 run_tleap.sh
    !bash run_tleap.sh

    # Volumeから0.15M相当のイオン数を計算
    !grep "Volume:" leap.log > temp.txt

    vol = None
    with open("temp.txt", 'r') as f:
      for line in f:
            vol = float(line.split()[1])

    if vol is None:
      raise ValueError("leap.logからVolumeを取得できませんでした。")

    vol_lit  = vol * pow(10, -27)
    atom_lit = 9.03 * pow(10, 22)
    conc = float(Concentration)
    num_ion = int(vol_lit * (conc/0.15) * atom_lit)

    if Ions == "NaCl":
      cation = "Na+"
    else:
      cation = "K+"

    anion = "Cl-"

    print("Volume:", vol)
    print("num_ion:", num_ion)
    print("Salt:", cation, anion)

    # 2回目tleap：中和 + 塩追加して本番topology作成
    f = open(tleap, "w")
    f.write("""source """ + str(ff) + "\n"
    """source leaprc.gaff2
    source """  + str(water) + "\n"
    """loadamberparams """ + str(ligand_frcmod) + "\n"
    """loadoff """ + str(lib) + "\n"
    """SYS = loadpdb """ + str(protein_ligand) + "\n"
    """alignaxes SYS
    savepdb SYS """ + str(pdb_nw) + "\n"
    """saveamberparm SYS """ + str(top_nw) + " " + str(crd_nw) + "\n"
    """solvatebox SYS """ + str(water_box) + " " + str(size_box) +  "  0.7\n"
    """addionsrand SYS Cl- 0\n"""
    """addionsrand SYS """ + str(cation) + " " + str(num_ion) + "\n"
    """addionsrand SYS """ + str(anion) + " " + str(num_ion) + "\n"
    """charge SYS\n"""
    """saveamberparm SYS """ + str(top) + " " + str(crd) + "\n"
    """savepdb SYS """ + str(pdb) + "\n"
    """quit""")
    f.close()

    tleap_command = "tleap -f " + str(tleap)

    with open('run_tleap.sh', 'w') as f:
        print(tleap_command, file=f)

    !rm -f SYS_gaff2.prmtop SYS_gaff2.crd SYS.pdb

    !chmod 700 run_tleap.sh
    !bash run_tleap.sh

    # final check
    pdb_amber = os.path.exists(pdb)
    top_amber = os.path.exists(top)
    crd_amber = os.path.exists(crd)

    if pdb_amber == True and top_amber == True and crd_amber == True:
      print("Successfully generated topology! :-)")
    else:
      print("ERROR: Check your inputs! ")

    print("\n重要警告チェック:")
    !grep -E "Errors|Warnings|There is a bond|One sided connection|unperturbed charge|Volume|Added|charge" leap.log || true

    !rm -f *.sh ANTECHAMBER* ATOMTYPE* temp.txt >/dev/null 2>&1



# ============================================================
# 5. Show initial 3D structure
# ============================================================

if RUN_SHOW_INITIAL_STRUCTURE:
    print('\n=== 5. Show initial 3D structure ===')
    # --- Original notebook cell 22 ---
    #chatgpt修正版

    #@title **Show 3D structure**
    import warnings
    warnings.filterwarnings('ignore')
    import py3Dmol
    import os

    color = "gray" #@param ["gray", "rainbow"]
    show_sidechains = False #@param {type:"boolean"}
    show_mainchains = False #@param {type:"boolean"}
    show_ligand = True #@param {type:"boolean"}
    show_box = True #@param {type:"boolean"}
    box_opacity = 0.6 #@param {type:"slider", min:0, max:1, step:0.1}

    pdb_file = os.path.join(workDir, "SYS.pdb")

    def show_pdb(show_sidechains=False, show_mainchains=False, show_ligand=False, show_box=False, color="rainbow"):
      view = py3Dmol.view(width=800, height=600)
      view.addModel(open(pdb_file, 'r').read(), 'pdb')

      if color == "gray":
        view.setStyle({'cartoon': {}})
      elif color == "rainbow":
        view.setStyle({'cartoon': {'color': 'spectrum'}})

      if show_sidechains:
        BB = ['C', 'O', 'N']
        view.addStyle(
            {'and':[{'resn':["GLY","PRO"], 'invert':True}, {'atom':BB, 'invert':True}]},
            {'stick':{'colorscheme':"WhiteCarbon", 'radius':0.3}}
        )
        view.addStyle(
            {'and':[{'resn':"GLY"}, {'atom':'CA'}]},
            {'sphere':{'colorscheme':"WhiteCarbon", 'radius':0.3}}
        )
        view.addStyle(
            {'and':[{'resn':"PRO"}, {'atom':['C','O'], 'invert':True}]},
            {'stick':{'colorscheme':"WhiteCarbon", 'radius':0.3}}
        )

      if show_mainchains:
        BB = ['C', 'O', 'N', 'CA']
        view.addStyle({'atom':BB}, {'stick':{'colorscheme':"WhiteCarbon", 'radius':0.3}})

      if show_box:
        view.addSurface(py3Dmol.SAS, {'opacity': box_opacity, 'color':'white'})

      if show_ligand:
        view.addStyle({'resn':'LIG'}, {'stick':{'colorscheme':'greenCarbon', 'radius':0.3}})
        view.setViewStyle({'style':'outline', 'color':'black', 'width':0.1})

      view.zoomTo()
      return view

    show_pdb(show_sidechains, show_mainchains, show_ligand, show_box, color).show()



# ============================================================
# 6. Initial LigPlot
# ============================================================

if RUN_INITIAL_LIGPLOT:
    print('\n=== 6. Initial LigPlot ===')
    # --- Original notebook cell 23 ---
    #chatgpt 修正版
    #@title **View and check the Ligand Interaction Network (LigPlot)**
    #@markdown This diagram is interactive and allows moving around the residues, as well as clicking the legend to toggle the display of specific residues types or interactions. The diagram will be saved as an HTML file (initial.html).

    import MDAnalysis as mda
    import prolif as plf
    import numpy as np
    import os
    from prolif.plotting.network import LigNetwork

    topology_file = os.path.join(workDir, "SYS_gaff2.prmtop")
    structure_file = os.path.join(workDir, "SYS.pdb")

    # load topology and coordinates
    u = mda.Universe(topology_file, structure_file)

    lig = u.select_atoms("resname LIG")
    prot = u.select_atoms("protein")

    print("Ligand atoms:", len(lig))
    print("Protein atoms:", len(prot))

    if len(lig) == 0:
        raise ValueError("LIG が見つかりません。SYS.pdb 内のリガンド残基名を確認してください。")

    if len(prot) == 0:
        raise ValueError("protein が見つかりません。MDAnalysisのprotein選択が空です。")

    # create RDKit-like molecules for visualisation
    lmol = plf.Molecule.from_mda(lig)
    pmol = plf.Molecule.from_mda(prot)

    fp = plf.Fingerprint()
    fp.run(u.trajectory[::10], lig, prot)
    df = fp.to_dataframe(return_atoms=True)

    net = LigNetwork.from_ifp(
        df,
        lmol,
        kind="frame",
        frame=0,
        rotation=270
    )

    net.save(os.path.join(workDir, "initial.html"))
    net.display()



# ============================================================
# 7. Equilibration MD
# ============================================================

if RUN_EQUILIBRATION:
    print('\n=== 7. Equilibration MD ===')
    # --- Original notebook cell 25 ---
    #chatgpt修正用
    #@title ### **Parameters for MD Equilibration protocol:**

    Jobname = 'prot_lig_equil' #@param {type:"string"}
    Jobname = Jobname.replace(" ", "_")

    top = os.path.join(workDir, "SYS_gaff2.prmtop")
    crd = os.path.join(workDir, "SYS_gaff2.crd")
    pdb = os.path.join(workDir, "SYS.pdb")

    Minimization_steps = "1000" #@param ["1000", "5000", "10000", "20000", "50000", "100000"]

    Time = "0.02" #@param {type:"string"}
    stride_time_eq = Time

    Integration_timestep = "2" #@param ["0.5", "1", "2", "3", "4"]
    dt_eq = Integration_timestep

    Temperature = "310" #@param {type:"string"}
    temperature_eq = Temperature

    Pressure = "1" #@param {type:"string"}
    pressure_eq = Pressure

    Force_constant = 700 #@param {type:"slider", min:0, max:2000, step:100}

    Write_the_trajectory = "10" #@param ["10", "100", "200", "500", "1000"]
    write_the_trajectory_eq = Write_the_trajectory

    Write_the_log = "10" #@param ["10", "100", "200", "500", "1000"]
    write_the_log_eq = Write_the_log

    # --- Original notebook cell 26 ---
    #chatgpt修正版
    #@title **Runs an Equilibration MD simulation (NPT ensemble)**
    #@markdown Now, let's equilibrate our system!

    ###########################################
    import openmm as mm
    from openmm import *
    from openmm.app import *
    from openmm.unit import *
    import pytraj as pt

    from sys import stdout, exit, stderr
    import os, math, fnmatch

    #############################################
    # Defining MD simulation parameters

    jobname = os.path.join(workDir, Jobname)
    coordinatefile = crd
    pdbfile = pdb
    topologyfile = top

    time_ps = float(Time)*1000
    simulation_time = float(time_ps)*picosecond		# in ps
    dt = int(dt_eq)*femtosecond
    temperature = float(temperature_eq)*kelvin
    savcrd_freq = int(write_the_trajectory_eq)*picosecond
    print_freq  = int(write_the_log_eq)*picosecond

    pressure	= float(pressure_eq)*bar

    restraint_fc = int(Force_constant) # kJ/mol

    nsteps  = int(simulation_time.value_in_unit(picosecond)/dt.value_in_unit(picosecond))
    nprint  = int(print_freq.value_in_unit(picosecond)/dt.value_in_unit(picosecond))
    nsavcrd = int(savcrd_freq.value_in_unit(picosecond)/dt.value_in_unit(picosecond))

    #############################################
    # Defining functions to use below:
    def backup_old_log(pattern, string):
    	result = []
    	for root, dirs, files in os.walk("./"):
    		for name in files:
    			if fnmatch.fnmatch(name, pattern):

    				try:
    					number = int(name[-2])
    					avail = isinstance(number, int)
    					#print(name,avail)
    					if avail == True:
    						result.append(number)
    				except:
    					pass

    	if len(result) > 0:
    		maxnumber = max(result)
    	else:
    		maxnumber = 0

    	backup_file = "\#" + string + "." + str(maxnumber + 1) + "#"
    	os.system("mv " + string + " " + backup_file)
    	return backup_file

    def restraints(system, crd, fc, restraint_array):

    	boxlx = system.getDefaultPeriodicBoxVectors()[0][0].value_in_unit(nanometers)
    	boxly = system.getDefaultPeriodicBoxVectors()[1][1].value_in_unit(nanometers)
    	boxlz = system.getDefaultPeriodicBoxVectors()[2][2].value_in_unit(nanometers)

    	if fc > 0:
    		# positional restraints for all heavy-atoms
    		posresPROT = CustomExternalForce('k*periodicdistance(x, y, z, x0, y0, z0)^2;')
    		posresPROT.addPerParticleParameter('k')
    		posresPROT.addPerParticleParameter('x0')
    		posresPROT.addPerParticleParameter('y0')
    		posresPROT.addPerParticleParameter('z0')

    		for atom1 in restraint_array:
    			atom1 = int(atom1)

    			xpos  = crd.positions[atom1].value_in_unit(nanometers)[0]
    			ypos  = crd.positions[atom1].value_in_unit(nanometers)[1]
    			zpos  = crd.positions[atom1].value_in_unit(nanometers)[2]

    			posresPROT.addParticle(atom1, [fc, xpos, ypos, zpos])

    		system.addForce(posresPROT)

    	return system
    ##############################################

    #############################################
    print("\n> Simulation details:\n")
    print("\tJob name = " + jobname)
    print("\tCoordinate file = " + str(coordinatefile))
    print("\tPDB file = " + str(pdbfile))
    print("\tTopology file = " + str(topologyfile))

    print("\n\tSimulation_time = " + str(simulation_time))
    print("\tIntegration timestep = " + str(dt))
    print("\tTotal number of steps = " +  str(nsteps))

    print("\n\tSave coordinates each " + str(savcrd_freq))
    print("\tPrint in log file each " + str(print_freq))

    print("\n\tTemperature = " + str(temperature))
    print("\tPressure = " + str(pressure))
    #############################################

    print("\n> Setting the system:\n")

    if Ligand_Force_field == "OpenFF 2.0.0 (Sage)":
      print("\t- Reading topology and structure file...")
      prmtop = pmd.load_file(topologyfile)
      inpcrd = AmberInpcrdFile(coordinatefile)

      print("\t- Creating system and setting parameters...")
      nonbondedMethod = PME
      nonbondedCutoff = 1.0*nanometers
      ewaldErrorTolerance = 0.0005
      constraints = HBonds
      rigidWater = True
      constraintTolerance = 0.000001
      friction = 1.0
      system = complex_structure.createSystem(nonbondedMethod=nonbondedMethod, nonbondedCutoff=nonbondedCutoff,
                                              constraints=constraints, rigidWater=rigidWater, ewaldErrorTolerance=ewaldErrorTolerance)
    else:
      print("\t- Reading topology and structure file...")
      prmtop = AmberPrmtopFile(topologyfile)
      inpcrd = AmberInpcrdFile(coordinatefile)

      print("\t- Creating system and setting parameters...")
      nonbondedMethod = PME
      nonbondedCutoff = 1.0*nanometers
      ewaldErrorTolerance = 0.0005
      constraints = HBonds
      rigidWater = True
      constraintTolerance = 0.000001
      friction = 1.0
      system = prmtop.createSystem(nonbondedMethod=nonbondedMethod, nonbondedCutoff=nonbondedCutoff,
                                              constraints=constraints, rigidWater=rigidWater, ewaldErrorTolerance=ewaldErrorTolerance)


    print("\t- Applying restraints. Force Constant = " + str(Force_constant) + "kJ/mol")
    pt_system = pt.iterload(coordinatefile, topologyfile)
    pt_topology = pt_system.top
    restraint_array = pt.select_atoms('!(:H*) & !(:WAT) & !(:Na+) & !(:Cl-) & !(:Mg+) & !(:K+)', pt_topology)

    system = restraints(system, inpcrd, restraint_fc, restraint_array)

    print("\t- Setting barostat...")
    system.addForce(MonteCarloBarostat(pressure, temperature))

    print("\t- Setting integrator...")
    integrator = LangevinIntegrator(temperature, friction, dt)
    integrator.setConstraintTolerance(constraintTolerance)
    simulation = Simulation(prmtop.topology, system, integrator)
    print("Platform:", simulation.context.getPlatform().getName())
    simulation.context.setPositions(inpcrd.positions)
    if inpcrd.boxVectors is not None:
        simulation.context.setPeriodicBoxVectors(*inpcrd.boxVectors)

    print("\t- Energy minimization: " + str(Minimization_steps) + " steps")
    simulation.minimizeEnergy(tolerance=10*kilojoule/mole/nanometer, maxIterations=int(Minimization_steps))

    print("\t-> Potential Energy = " + str(simulation.context.getState(getEnergy=True).getPotentialEnergy()))

    print("\t- Setting initial velocities...")
    simulation.context.setVelocitiesToTemperature(temperature)

    #############################################
    # Running Equilibration on NPT ensemble

    dcd_file = jobname + ".dcd"
    log_file = jobname + ".log"
    rst_file = jobname + ".rst"
    prv_rst_file = jobname + ".rst"
    pdb_file = jobname + ".pdb"

    # Creating a trajectory file and reporters
    dcd = DCDReporter(dcd_file, nsavcrd)
    firstdcdstep = (nsteps) + nsavcrd
    dcd._dcd = DCDFile(dcd._out, simulation.topology, simulation.integrator.getStepSize(), firstdcdstep, nsavcrd) # charmm doesn't like first step to be 0

    simulation.reporters.append(dcd)
    simulation.reporters.append(StateDataReporter(stdout, nprint, step=True, speed=True, progress=True, totalSteps=nsteps, remainingTime=True, separator='\t\t'))
    simulation.reporters.append(StateDataReporter(log_file, nprint, step=True, kineticEnergy=True, potentialEnergy=True, totalEnergy=True, temperature=True, volume=True, speed=True))

    print("\n> Simulating " + str(nsteps) + " steps...")
    simulation.step(nsteps)

    simulation.reporters.clear() # remove all reporters so the next iteration don't trigger them.


    ##################################
    # Writing last frame information of stride
    print("\n> Writing state file (" + str(rst_file) + ")...")
    state = simulation.context.getState( getPositions=True, getVelocities=True )
    with open(rst_file, 'w') as f:
    	f.write(XmlSerializer.serialize(state))

    last_frame = int(nsteps/nsavcrd)
    print("> Writing coordinate file (" + str(pdb_file) + ", frame = " + str(last_frame) + ")...")
    positions = simulation.context.getState(getPositions=True).getPositions()
    PDBFile.writeFile(simulation.topology, positions, open(pdb_file, 'w'))

    print("\n> Finished!\n")



# ============================================================
# 8. Production MD
# ============================================================

if RUN_PRODUCTION:
    print('\n=== 8. Production MD ===')
    # --- Original notebook cell 28 ---
    #chatgpt修正版

    #@markdown ### **Provide input file names below:**

    Equilibrated_PDB = 'prot_lig_equil.pdb' #@param {type:"string"}
    State_file = 'prot_lig_equil.rst' #@param {type:"string"}
    Ligand_Force_field = "GAFF2"

    #@markdown ---
    #@markdown ### **Parameters for MD Production protocol:**

    Jobname = 'prot_lig_prod' #@param {type:"string"}
    Jobname = Jobname.replace(" ", "_")

    top = os.path.join(workDir, "SYS_gaff2.prmtop")
    crd = os.path.join(workDir, "SYS_gaff2.crd")
    pdb = os.path.join(workDir, "SYS.pdb")

    Stride_Time = "0.01" #@param {type:"string"}
    stride_time_prod = Stride_Time

    Number_of_strides = "30" #@param {type:"string"}
    nstride = Number_of_strides

    Integration_timestep = "2" #@param ["0.5", "1", "2", "3", "4"]
    dt_prod = Integration_timestep

    Temperature = "310" #@param {type:"string"}
    temperature_prod = Temperature

    Pressure = "1" #@param {type:"string"}
    pressure_prod = Pressure

    Write_the_trajectory = "10" #@param ["10", "100", "200", "500", "1000"]
    write_the_trajectory_prod = Write_the_trajectory

    Write_the_log = "10" #@param ["10", "100", "200", "500", "1000"]
    write_the_log_prod = Write_the_log

    # --- Original notebook cell 29 ---
    #chatgpt修正版

    #@title **Runs a Production MD simulation (NPT ensemble) after equilibration**
    #
    ###########################################
    import openmm as mm
    from openmm import *
    from openmm.app import *
    from openmm.unit import *

    from sys import stdout, exit, stderr
    import os, math, fnmatch

    #############################################
    # Defining MD simulation parameters

    jobname = os.path.join(workDir, str(Jobname))
    coordinatefile = crd
    pdbfile = os.path.join(workDir, Equilibrated_PDB)
    topologyfile = top
    equil_rst_file = os.path.join(workDir, State_file)


    stride_time_ps = float(stride_time_prod)*1000
    stride_time = float(stride_time_ps)*picosecond
    nstride = int(Number_of_strides)
    dt = int(dt_prod)*femtosecond
    temperature = float(temperature_prod)*kelvin
    savcrd_freq = int(write_the_trajectory_prod)*picosecond
    print_freq  = int(write_the_log_prod)*picosecond

    pressure	= float(pressure_prod)*bar

    simulation_time = stride_time*nstride
    nsteps  = int(stride_time.value_in_unit(picosecond)/dt.value_in_unit(picosecond))
    nprint  = int(print_freq.value_in_unit(picosecond)/dt.value_in_unit(picosecond))
    nsavcrd = int(savcrd_freq.value_in_unit(picosecond)/dt.value_in_unit(picosecond))
    firststride = 1 # must be integer
    #############################################
    # Defining functions to use below:
    def backup_old_log(pattern, string):
    	result = []
    	for root, dirs, files in os.walk("./"):
    		for name in files:
    			if fnmatch.fnmatch(name, pattern):

    				try:
    					number = int(name[-2])
    					avail = isinstance(number, int)
    					#print(name,avail)
    					if avail == True:
    						result.append(number)
    				except:
    					pass

    	if len(result) > 0:
    		maxnumber = max(result)
    	else:
    		maxnumber = 0

    	backup_file = "\#" + string + "." + str(maxnumber + 1) + "#"
    	os.system("mv " + string + " " + backup_file)
    	return backup_file
    ##############################################

    #############################################
    print("\n> Simulation details:\n")
    print("\tJob name = " + jobname)
    print("\tCoordinate file = " + str(coordinatefile))
    print("\tPDB file = " + str(pdbfile))
    print("\tTopology file = " + str(topologyfile))

    print("\n\tSimulation_time = " + str(stride_time*nstride))
    print("\tIntegration timestep = " + str(dt))
    print("\tTotal number of steps = " +  str(nsteps*nstride))
    print("\tNumber of strides = " + str(nstride) + " (" + str(stride_time) + " in each stride)")

    print("\n\tSave coordinates each " + str(savcrd_freq))
    print("\tSave checkpoint each " + str(savcrd_freq))
    print("\tPrint in log file each " + str(print_freq))

    print("\n\tTemperature = " + str(temperature))
    print("\tPressure = " + str(pressure))
    #############################################

    print("\n> Setting the system:\n")

    if Ligand_Force_field == "OpenFF 2.0.0 (Sage)":
      print("\t- Reading topology and structure file...")
      prmtop = pmd.load_file(topologyfile)
      inpcrd = AmberInpcrdFile(coordinatefile)

      print("\t- Creating system and setting parameters...")
      nonbondedMethod = PME
      nonbondedCutoff = 1.0*nanometers
      ewaldErrorTolerance = 0.0005
      constraints = HBonds
      rigidWater = True
      constraintTolerance = 0.000001
      friction = 1.0
      system = complex_structure.createSystem(nonbondedMethod=nonbondedMethod, nonbondedCutoff=nonbondedCutoff,
                                              constraints=constraints, rigidWater=rigidWater, ewaldErrorTolerance=ewaldErrorTolerance)
    else:
      print("\t- Reading topology and structure file...")
      prmtop = AmberPrmtopFile(topologyfile)
      inpcrd = AmberInpcrdFile(coordinatefile)

      print("\t- Creating system and setting parameters...")
      nonbondedMethod = PME
      nonbondedCutoff = 1.0*nanometers
      ewaldErrorTolerance = 0.0005
      constraints = HBonds
      rigidWater = True
      constraintTolerance = 0.000001
      friction = 1.0
      system = prmtop.createSystem(nonbondedMethod=nonbondedMethod, nonbondedCutoff=nonbondedCutoff,
                                              constraints=constraints, rigidWater=rigidWater, ewaldErrorTolerance=ewaldErrorTolerance)

    print("\t- Setting barostat...")
    system.addForce(MonteCarloBarostat(pressure, temperature))

    print("\t- Setting integrator...")
    integrator = LangevinIntegrator(temperature, friction, dt)
    integrator.setConstraintTolerance(constraintTolerance)
    simulation = Simulation(prmtop.topology, system, integrator)
    print("Platform:", simulation.context.getPlatform().getName())
    simulation.context.setPositions(inpcrd.positions)
    if inpcrd.boxVectors is not None:
    	simulation.context.setPeriodicBoxVectors(*inpcrd.boxVectors)

    #############################################
    # Opening a loop of extension NSTRIDE to simulate the entire STRIDE_TIME*NSTRIDE
    for n in range(1, nstride + 1):

    	print("\n\n>>> Simulating Stride #" + str(n) + " <<<")

    	dcd_file = jobname + "_" + str(n) + ".dcd"
    	log_file = jobname + "_" + str(n) + ".log"
    	rst_file = jobname + "_" + str(n) + ".rst"
    	prv_rst_file = jobname + "_" + str(n-1) + ".rst"
    	pdb_file = jobname + "_" + str(n) + ".pdb"

    	if os.path.exists(rst_file):
    		print("> Stride #" + str(n) + " finished (" + rst_file + " present). Moving to next stride... <")
    		continue

    	if n == 1:
    		print("\n> Loading previous state from equilibration > " + equil_rst_file + " <")
    		with open(equil_rst_file, 'r') as f:
    			simulation.context.setState(XmlSerializer.deserialize(f.read()))
    			currstep = int((n-1)*nsteps)
    			currtime = currstep*dt.in_units_of(picosecond)
    			simulation.currentStep = currstep
    			simulation.context.setTime(currtime)
    			print("> Current time: " + str(currtime) + " (Step = " + str(currstep) + ")")

    	else:
    		print("> Loading previous state from > " + prv_rst_file + " <")
    		with open(prv_rst_file, 'r') as f:
    			simulation.context.setState(XmlSerializer.deserialize(f.read()))
    			currstep = int((n-1)*nsteps)
    			currtime = currstep*dt.in_units_of(picosecond)
    			simulation.currentStep = currstep
    			simulation.context.setTime(currtime)
    			print("> Current time: " + str(currtime) + " (Step = " + str(currstep) + ")")


    	dcd = DCDReporter(dcd_file, nsavcrd)
    	firstdcdstep = (currstep) + nsavcrd
    	dcd._dcd = DCDFile(dcd._out, simulation.topology, simulation.integrator.getStepSize(), firstdcdstep, nsavcrd) # first step should not be 0

    	simulation.reporters.append(dcd)
    	simulation.reporters.append(StateDataReporter(stdout, nprint, step=True, speed=True, progress=True, totalSteps=(nsteps*nstride), remainingTime=True, separator='\t\t'))
    	simulation.reporters.append(StateDataReporter(log_file, nprint, step=True, kineticEnergy=True, potentialEnergy=True, totalEnergy=True, temperature=True, volume=True, speed=True))

    	print("\n> Simulating " + str(nsteps) + " steps... (Stride #" + str(n) + ")")
    	simulation.step(nsteps)

    	simulation.reporters.clear() # remove all reporters so the next iteration don't trigger them.


    	##################################
    	# Writing last frame information of stride
    	print("\n> Writing state file (" + str(rst_file) + ")...")
    	state = simulation.context.getState( getPositions=True, getVelocities=True )
    	with open(rst_file, 'w') as f:
    		f.write(XmlSerializer.serialize(state))

    	last_frame = int(nsteps/nsavcrd)
    	print("> Writing coordinate file (" + str(pdb_file) + ", frame = " + str(last_frame) + ")...")
    	positions = simulation.context.getState(getPositions=True).getPositions()
    	PDBFile.writeFile(simulation.topology, positions, open(pdb_file, 'w'))

    print("\n> Finished!\n")



# ============================================================
# 9. Concatenate and align trajectory
# ============================================================

if RUN_CONCAT_TRAJECTORY:
    print('\n=== 9. Concatenate and align trajectory ===')
    # --- Original notebook cell 30 ---
    #chatgpt修正版

    #@title **Concatenate and align the trajectory**
    #@markdown **Important**: Jobname, Number of strides, stride time and trajectory saved frequency should match the production run.

    import os
    import MDAnalysis as mda
    from MDAnalysis.analysis import align, rms

    # Google Drive は使わず、GPUサーバー側の作業ディレクトリを使う
    workDir = "/home/gregorymaddux18/making_it_rain_work"

    Equilibrated_PDB = 'prot_lig_equil.pdb' #@param {type:"string"}
    Jobname = "prot_lig_prod" #@param {type: "string"}

    Skip = "1" #@param ["1", "2", "5", "10", "20", "50"]
    Skip = int(Skip)
    stride_traj = Skip

    Output_format = "dcd" #@param ["dcd", "pdb", "trr", "xtc"]

    first_stride = "1" #@param {type:"string"}

    Number_of_strides = "30" #@param {type:"string"}
    nstride = int(Number_of_strides)

    stride_time = "0.01" #@param {type:"string"}

    trajectory_saved_frequency = "10" #@param ["10", "100", "200", "500", "1000"]
    traj_save_freq = trajectory_saved_frequency

    Remove_waters = "no" #@param ["yes", "no"]

    output_prefix = first_stride + "-" + str(int(first_stride) + nstride - 1)

    stride_time_ps = float(stride_time) * 1000
    simulation_time_analysis = stride_time_ps * nstride
    simulation_ns = float(stride_time) * int(Number_of_strides)
    number_frames = int(simulation_time_analysis) / int(traj_save_freq)
    number_frames_analysis = number_frames / int(Skip)

    nw_dcd = os.path.join(workDir, str(Jobname) + output_prefix + "_nw." + str(Output_format))
    whole_dcd = os.path.join(workDir, str(Jobname) + output_prefix + "_whole." + str(Output_format))

    template = os.path.join(workDir, str(Jobname) + '_%s.dcd')
    pdb = os.path.join(workDir, Equilibrated_PDB)

    flist = [template % str(i) for i in range(int(first_stride), int(first_stride) + nstride)]

    print("Trajectory files check:")
    missing_files = []

    for f in flist:
        if not os.path.exists(f):
            missing_files.append(f)

    if len(missing_files) > 0:
        print("Missing files:")
        for f in missing_files:
            print(f)
        raise FileNotFoundError("一部のproduction DCDファイルが見つかりません。")
    else:
        print("All DCD files found.")

    print("Expected total simulation time:", simulation_ns, "ns")
    print("Expected frames before skip:", number_frames)
    print("Expected frames after skip:", number_frames_analysis)

    if Remove_waters == "yes":
      # Save topology without waters
      gaff_top = pt.load_topology(os.path.join(workDir, "SYS_gaff2.prmtop"))
      gaff_nw = gaff_top['!:WAT']
      gaff_nw.save(os.path.join(workDir, "SYS_gaff2_nw.prmtop"))

      # Save trajectory without waters
      trajlist = pt.load(flist, os.path.join(workDir, "SYS_gaff2.prmtop"), stride=Skip)
      t0 = trajlist.strip(':WAT')
      traj_image = t0.iterframe(autoimage=True, rmsfit=0)
      pt.write_traj(nw_dcd, traj_image, overwrite=True, options=Output_format)

      traj_dcd_check = os.path.exists(nw_dcd)
      traj = nw_dcd
      pdb_ref = os.path.join(workDir, "SYS_gaff2_nw.prmtop")

    else:
      trajlist = pt.load(flist, os.path.join(workDir, "SYS_gaff2.prmtop"), stride=Skip)
      traj_image = trajlist.iterframe(autoimage=True, rmsfit=0)
      pt.write_traj(whole_dcd, traj_image, overwrite=True, options=Output_format)

      traj_dcd_check = os.path.exists(whole_dcd)
      traj = whole_dcd
      pdb_ref = os.path.join(workDir, "SYS_gaff2.prmtop")

    traj_load = pt.load(traj, pdb_ref)
    print(traj_load)

    if traj_dcd_check == True:
      print("Trajectory concatenated successfully! :-)")
      print("Output trajectory:", traj)
    else:
      print("ERROR: Check your inputs!")



# ============================================================
# 10. Load, view and check trajectory
# ============================================================

if RUN_VIEW_TRAJECTORY:
    print('\n=== 10. Load, view and check trajectory ===')
    # --- Original notebook cell 31 ---
    #chatgpt修正版

    #@title **Load, view and check the trajectory**
    #@markdown This will take a few minutes. Another coffee would be great. :-)

    import warnings
    warnings.filterwarnings('ignore')

    #py3dmol functions
    class Atom(dict):
      def __init__(self, line):
        self["type"] = line[0:6].strip()
        self["idx"] = line[6:11].strip()
        self["name"] = line[12:16].strip()
        self["resname"] = line[17:20].strip()
        self["resid"] = int(int(line[22:26]))
        self["x"] = float(line[30:38])
        self["y"] = float(line[38:46])
        self["z"] = float(line[46:54])
        self["sym"] = line[76:78].strip()

      def __str__(self):
        line = list(" " * 80)
        line[0:6] = self["type"].ljust(6)
        line[6:11] = self["idx"].ljust(5)
        line[12:16] = self["name"].ljust(4)
        line[17:20] = self["resname"].ljust(3)
        line[22:26] = str(self["resid"]).ljust(4)
        line[30:38] = str(self["x"]).rjust(8)
        line[38:46] = str(self["y"]).rjust(8)
        line[46:54] = str(self["z"]).rjust(8)
        line[76:78] = self["sym"].rjust(2)
        return "".join(line) + "\n"

    class Molecule(list):
      def __init__(self, file):
        for line in file:
          if "ATOM" in line or "HETATM" in line:
            self.append(Atom(line))

        def __str__(self):
          outstr = ""
          for at in self:
            outstr += str(at)
          return outstr

    if number_frames_analysis > 10:
      stride_animation = number_frames_analysis/10
    else:
      stride_animation = 1

    u = mda.Universe(pdb_ref, traj)

    # Write out frames for animation
    protein = u.select_atoms('not (resname WAT)')
    i = 0
    for ts in u.trajectory[0:len(u.trajectory):int(stride_animation)]:
        if i > -1:
            with mda.Writer('' + str(i) + '.pdb', protein.n_atoms) as W:
                W.write(protein)
        i = i + 1
    # Load frames as molecules
    molecules = []
    for i in range(int(len(u.trajectory)/int(stride_animation))):
        with open('' + str(i) + '.pdb') as ifile:
            molecules.append(Molecule(ifile))

    models = ""
    for i in range(len(molecules)):
      models += "MODEL " + str(i) + "\n"
      for j,mol in enumerate(molecules[i]):
        models += str(mol)
      models += "ENDMDL\n"
    #view.addModelsAsFrames(models)

    # Animation
    view = py3Dmol.view(width=800, height=600)
    view.addModelsAsFrames(models)
    for i, at in enumerate(molecules[0]):
        default = {"cartoon": {'color': 'spectrum'}}
        view.setViewStyle({'style':'outline','color':'black','width':0.1})
        view.setStyle({'model': -1, 'serial': i+1}, at.get("pymol", default))
        HP = ['LIG']
        view.setStyle({"model":-1,'and':[{'resn':HP}]},{'stick':{'radius':0.3}})
    view.zoomTo()
    view.animate({'loop': "forward"})
    view.show()



# ============================================================
# 11. MD LigPlot
# ============================================================

if RUN_MD_LIGPLOT:
    print('\n=== 11. MD LigPlot ===')
    # --- Original notebook cell 32 ---
    #chatgpt修正版

    #@title **View and check the Ligand Interaction Network (LigPlot) during MD simulations**
    #@markdown This diagram is interactive and allows moving around the residues, as well as clicking the legend to toggle the display of specific residues types or interactions. The diagram will be saved as an HTML file (output.html).

    #@markdown **Provide output file names below:**
    Output_name = 'Interaction' #@param {type:"string"}

    #@markdown The frequency with which an interaction is seen will control the width of the corresponding edge. You can hide the least frequent interactions by using a threshold, i.e. threshold=0.3 will hide interactions that occur in less than 30% of frames.
    Threshold = 0.3 #@param {type:"slider", min:0, max:1.0, step:0.1}

    import MDAnalysis as mda
    import prolif as plf
    import numpy as np
    import os
    from prolif.plotting.network import LigNetwork

    # load topology
    u = mda.Universe(pdb_ref, traj)
    lig = u.select_atoms("resname LIG")
    prot = u.select_atoms("protein")

    # create RDKit-like molecules for visualisation
    lmol = plf.Molecule.from_mda(lig)
    pmol = plf.Molecule.from_mda(prot)

    if number_frames_analysis > 10:
      stride_animation = number_frames_analysis/10
    else:
      stride_animation = 1

    fp = plf.Fingerprint()
    #fp.run(u.trajectory[::int(stride_animation)], lig, prot)
    # スライスを使わずに、trajectory全体を渡す形に変える
    fp.run(u.trajectory, lig, prot)
    df = fp.to_dataframe(return_atoms=True)

    net = LigNetwork.from_ifp(df, lmol,
                              # replace with `kind="frame", frame=0` for the other depiction
                              kind="aggregate", threshold=float(Threshold),
                              rotation=270)
    net.save(os.path.join(workDir, Output_name + ".html"))
    net.display()



# ============================================================
# 12. MM-PBSA / MM-GBSA
# ============================================================

if RUN_MMPBSA:
    print('\n=== 12. MM-PBSA / MM-GBSA ===')
    # --- Original notebook cell 34 ---
    #chatgpt 修正版

    #@title **MM-PBSA method to calculate the binding free energy**
    #@markdown **Important:** We will now calculate the interaction energy and solvation free energy for the complex, receptor and ligand and average the results to obtain an estimate of the binding free energy.

    igb = "2" #@param ["1", "2", "5", "7", "8"]

    import os
    import sys
    import locale

    def getpreferredencoding(do_setlocale=True):
        return "UTF-8"

    locale.getpreferredencoding = getpreferredencoding

    if igb == "1":
      mbondi = 'mbondi'
    elif igb == "2" or igb == "5":
      mbondi = 'mbondi2'
    elif igb == "7":
      mbondi = 'bondi'
    elif igb == "8":
      mbondi = 'mbondi3'
    else:
      raise ValueError("igb の値が不正です。")

    Salt_concentration = '0.15' #@param {type:"string"}
    fold_MMPBSA = "MMPBSA_igb_" + igb

    #@markdown **Provide output file names below:**
    Output_name = 'FINAL_RESULTS_MMPBSA' #@param {type:"string"}

    final_mmpbsa = os.path.join(workDir, Output_name)

    if number_frames_analysis > 10:
      stride = number_frames_analysis / 10
    else:
      stride = 1

    stride = max(1, int(stride))

    # 水・イオンを除外するmask
    strip_mask = ":WAT,Na+,Cl-,Mg+,K+"

    print("pdb_ref:", pdb_ref)
    print("traj:", traj)
    print("number_frames_analysis:", number_frames_analysis)
    print("MMPBSA endframe:", int(number_frames_analysis))
    print("MMPBSA interval:", stride)
    print("strip_mask:", strip_mask)

    if not os.path.exists(pdb_ref):
        raise FileNotFoundError("pdb_ref が見つかりません: " + str(pdb_ref))

    if not os.path.exists(traj):
        raise FileNotFoundError("traj が見つかりません: " + str(traj))

    f = open("mmpbsa.in", "w")
    f.write("""&general
      endframe=""" + str(int(number_frames_analysis)) + """, interval=""" + str(stride) + """, strip_mask=""" + strip_mask + """,
    /
    &gb
      igb=""" + str(igb) + """, saltcon=""" + str(Salt_concentration) + """,
    /
    &pb
      istrng=""" + str(Salt_concentration) + """, inp=2, radiopt=0, prbrad=1.4,
    /
    """)
    f.close()

    ante_MMPBSA = (
        "ante-MMPBSA.py"
        + " -p " + str(pdb_ref)
        + " -c com.prmtop"
        + " -r rec.prmtop"
        + " -l ligand.prmtop"
        + " -s " + strip_mask
        + " -n :LIG"
        + " --radii " + str(mbondi)
    )

    MMPBSA = (
        "MMPBSA.py -O"
        + " -i mmpbsa.in"
        + " -o " + str(final_mmpbsa) + ".dat"
        + " -sp " + str(pdb_ref)
        + " -cp com.prmtop"
        + " -rp rec.prmtop"
        + " -lp ligand.prmtop"
        + " -y " + str(traj)
    )

    mkdir = "mkdir -p " + os.path.join(workDir, fold_MMPBSA)
    mv = "mv -f _MMPBSA* com.prmtop rec.prmtop ligand.prmtop reference.frc mmpbsa.in " + os.path.join(workDir, fold_MMPBSA)

    original_stdout = sys.stdout

    with open('run_MMPBSA.sh', 'w') as f:
        sys.stdout = f
        print(ante_MMPBSA)
        print(MMPBSA)
        print(mkdir)
        print(mv)
        sys.stdout = original_stdout

    !chmod 700 run_MMPBSA.sh

    # 初回はエラーを隠さず表示する
    !bash run_MMPBSA.sh

    if os.path.exists(final_mmpbsa + '.dat'):
        with open(final_mmpbsa + '.dat', 'r') as f_mmpbsa:
            file_contents = f_mmpbsa.read()
        print(file_contents)
    else:
        raise FileNotFoundError(final_mmpbsa + ".dat が作成されていません。上のMMPBSAエラーを確認してください。")



# ============================================================
# 12.5 Interaction energy debug
# ============================================================

if RUN_INTERACTION_ENERGY_DEBUG:
    print('\n=== 12.5 Interaction energy debug ===')
    # --- Original notebook cell 35 ---
    # エラーが出ている 47行目の直前にこれを差し込んで実行してみてください
    print(f"データ数: {len(filtered_lie_total)}")

    # もしデータ数が1なら、標準偏差の計算を飛ばすように書き換える（暫定処置）
    if len(filtered_lie_total) >= 2:
        lie_total_stdev = stdev(filtered_lie_total)
    else:
        lie_total_stdev = 0.0
        print("警告: データが足りないため標準偏差は0として表示します")



# ============================================================
# 13. Interaction Energy
# ============================================================

if RUN_INTERACTION_ENERGY:
    print('\n=== 13. Interaction Energy ===')
    # --- Original notebook cell 36 ---
    #chatgpt修正版

    #@title **Interaction Energy**
    #@markdown **Important:** To quantify the strength of the interaction between the ligand and the protein, we will compute the nonbonded interaction energy between these two species.  It is important to note that this quantity is NOT a free energy or a binding energy.

    #@markdown **Provide output file names below:**
    Output_name = 'Interaction_energy' #@param {type:"string"}

    pt_topology = traj_load.top
    restraint_array = pt.select_atoms('!(:WAT) & !(:Na+) & !(:Cl-) & !(:Mg+) & !(:K+) & !(:LIG)', pt_topology)
    first_atom = restraint_array[0]
    last_atom = restraint_array[-1]
    mask = "LIE :LIG @" + str(first_atom+1) + "-" + str(last_atom+1)

    lie = pt.analysis.energy_analysis.lie(traj_load, mask=mask, options='cutvdw 12.0 cutelec 12.0 diel 2.0', dtype='dict')

    lie_elec = lie['LIE[EELEC]']
    lie_vdw = lie['LIE[EVDW]']
    lie_total = lie_elec + lie_vdw
    Write_the_trajectory = traj_save_freq
    time = len(lie_total)*int(Write_the_trajectory)/1000
    time_array = np.arange(0,time,int(Write_the_trajectory)/1000)*int(stride_traj)

    def filter_outliers(data):
        """Return a mask of booleans to filter out outliers."""
        Q1 = np.percentile(data, 25)
        Q3 = np.percentile(data, 75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        return (data >= lower_bound) & (data <= upper_bound)

    # Create masks for non-outliers
    mask_total = filter_outliers(lie_total)
    mask_elec = filter_outliers(lie_elec)
    mask_vdw = filter_outliers(lie_vdw)

    # Filter values based on the mask
    filtered_time_total = time_array[mask_total]
    filtered_lie_total = lie_total[mask_total]

    filtered_time_elec = time_array[mask_elec]
    filtered_lie_elec = lie_elec[mask_elec]

    filtered_time_vdw = time_array[mask_vdw]
    filtered_lie_vdw = lie_vdw[mask_vdw]

    lie_total_mean = mean(filtered_lie_total)
    lie_total_stdev = stdev(filtered_lie_total)
    print("Interaction Energy Average = " + str("{:.2f}".format(lie_total_mean)) + " \u00B1 " + str("{:.2f}".format(lie_total_stdev)) + " kcal/mol")

    # Plotting:
    plt.plot(filtered_time_total, filtered_lie_total, alpha=0.6, color='blue', linewidth=1.5, label="Total Energy")
    plt.plot(filtered_time_elec, filtered_lie_elec, alpha=0.6, color='green', linewidth=1.5, label="Electrostatic Energy")
    plt.plot(filtered_time_vdw, filtered_lie_vdw, alpha=0.6, color='red', linewidth=1.5, label="van der Waals Energy")

    plt.xlim(0, simulation_ns)
    # plt.ylim(-50, 0)

    plt.xlabel("Time (ns)", fontsize=14, fontweight='bold')
    plt.ylabel('Interaction Energy \n (kcal/mol)', fontsize=14, fontweight='bold')
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.legend(frameon=False, loc='center left', bbox_to_anchor=(1, 0.5))
    plt.savefig(os.path.join(workDir, Output_name + ".png"), dpi=600, bbox_inches='tight')

    lie_eelec = pd.DataFrame(lie['LIE[EELEC]'])
    lie_eelec.to_csv(os.path.join(workDir, Output_name + "_eelec.csv"))
    lie_evdw = pd.DataFrame(lie['LIE[EVDW]'])
    lie_evdw.to_csv(os.path.join(workDir, Output_name + "_evdw.csv"))



# ============================================================
# 14. Distance to nearest residues
# ============================================================

if RUN_DISTANCE_NEAREST_RESIDUES:
    print('\n=== 14. Distance to nearest residues ===')
    # --- Original notebook cell 37 ---
    #chatgpt修正版

    #@title **Compute distance between the ligand and catalytic site residues**
    #@markdown **Provide output file names below:**
    Output_name = 'distance' #@param {type:"string"}

    #@markdown **Cutoff distance to nearest residues (Angstrons):**
    Distance = '5' #@param {type:"string"}

    top = pt_topology

    # 最初のフレームを基準に、LIGからDistance Å以内の非水・非イオン・非LIG残基を選ぶ
    top.set_reference(traj_load[0])

    indices = top.select(
        '(:LIG<:' + str(Distance) + ')&!(:WAT|:Na+|:Cl-|:Mg+|:K+|:LIG)'
    )

    residues = [res.original_resid for res in top[indices].residues]
    residues = sorted(set(residues))

    if len(residues) == 0:
        raise ValueError("LIGから " + str(Distance) + " Å 以内の残基が見つかりません。Distanceを大きくしてください。")

    res_string = ','.join(str(e) for e in residues)
    print("Selected residues = " + res_string + "\n")

    mask = ":LIG :" + str(res_string)
    dist = pt.distance(traj_load, mask)

    dist_mean = mean(dist)
    dist_stdev = stdev(dist)

    print(
        "Distance Average = "
        + str("{:.2f}".format(dist_mean))
        + " ± "
        + str("{:.2f}".format(dist_stdev))
        + " Å"
    )

    time = len(dist) * int(Write_the_trajectory) / 1000
    time_array = np.arange(0, time, int(Write_the_trajectory) / 1000) * int(stride_traj)

    # Plotting:
    plt.plot(time_array, dist, alpha=1, color='springgreen', linewidth=1.0)
    plt.xlim(0, simulation_ns)
    #plt.ylim(2, 6)

    plt.xlabel("Time (ns)", fontsize=14, fontweight='bold')
    plt.ylabel("Distance [$\AA$]", fontsize=14, fontweight='bold')
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    plt.savefig(os.path.join(workDir, Output_name + ".png"), dpi=600, bbox_inches='tight')
    plt.show()

    raw_data = pd.DataFrame(dist)
    raw_data.to_csv(os.path.join(workDir, Output_name + ".csv"))



# ============================================================
# 15. Minimum distance to selected residues
# ============================================================

if RUN_DISTANCE_SPECIFIC_RESIDUES_MIN:
    print('\n=== 15. Minimum distance to selected residues ===')
    # --- Original notebook cell 38 ---
    #chatgpt修正版 指定のアミノ酸から最も近いリガンドの部位との距離を
    #各フレームごとに算出

    #@title **Compute minimum distance between ligand carbons and specific residues**
    Output_name = 'distance_select_min_ligC' #@param {type:"string"}
    Residues = '109,168,173,182' #@param {type:"string"}

    residue_list = [r.strip() for r in Residues.split(",")]

    top = traj_load.top

    # リガンドの炭素原子だけを選択
    lig_c_indices = top.select(":LIG&@C*")

    if len(lig_c_indices) == 0:
        raise ValueError("LIGの炭素原子が見つかりません。原子名を確認してください。")

    distance_data = {}

    for res in residue_list:
        # 指定残基の重原子を選択。水素は除外
        res_indices = top.select(":" + res + "&!@H=")

        if len(res_indices) == 0:
            print("Warning: residue " + res + " が見つかりません。スキップします。")
            continue

        min_distances = []

        for frame in traj_load:
            xyz = frame.xyz

            lig_xyz = xyz[lig_c_indices]
            res_xyz = xyz[res_indices]

            # 全組み合わせの距離を計算
            diff = lig_xyz[:, None, :] - res_xyz[None, :, :]
            distances = np.sqrt(np.sum(diff * diff, axis=2))

            # そのフレームでの最短距離
            min_distances.append(np.min(distances))

        distance_data[res] = np.array(min_distances)

    # 時間軸
    time = len(next(iter(distance_data.values()))) * int(Write_the_trajectory) / 1000
    time_array = np.arange(0, time, int(Write_the_trajectory) / 1000) * int(stride_traj)

    # Plotting
    for res, dist in distance_data.items():
        dist_mean = mean(dist)
        dist_stdev = stdev(dist)
        print("Residue " + res + " minimum distance = " + "{:.2f}".format(dist_mean) + " ± " + "{:.2f}".format(dist_stdev) + " Å")
        plt.plot(time_array, dist, linewidth=1.0, label="Res " + res)

    plt.xlim(0, simulation_ns)
    plt.xlabel("Time (ns)", fontsize=14, fontweight='bold')
    plt.ylabel("Minimum distance to ligand carbons [$\AA$]", fontsize=14, fontweight='bold')
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.legend(frameon=False, loc='center left', bbox_to_anchor=(1, 0.5))

    plt.savefig(os.path.join(workDir, Output_name + ".png"), dpi=600, bbox_inches='tight')
    plt.show()

    # CSV保存
    raw_data = pd.DataFrame(distance_data)
    raw_data.insert(0, "Time_ns", time_array)
    raw_data.to_csv(os.path.join(workDir, Output_name + ".csv"), index=False)



# ============================================================
# 15.5 Detect nearby residues
# ============================================================

if RUN_DETECT_NEARBY_RESIDUES:
    print('\n=== 15.5 Detect nearby residues ===')
    # --- Original notebook cell 39 ---
    #chatgpt 修正版 近傍のアミノ酸を検出

    from MDAnalysis.lib.distances import distance_array
    import numpy as np

    u = mda.Universe(pdb_ref, traj)
    u.trajectory[0]

    box = u.dimensions
    lig_c = u.select_atoms("resname LIG and name C*")

    rows = []

    for res in u.select_atoms("protein").residues:
        res_atoms = res.atoms.select_atoms("not name H*")
        if len(res_atoms) == 0:
            continue

        dmat = distance_array(lig_c.positions, res_atoms.positions, box=box)
        i, j = np.unravel_index(np.argmin(dmat), dmat.shape)

        lig_atom = lig_c[i]
        res_atom = res_atoms[j]
        min_d = dmat[i, j]

        rows.append((min_d, res.resid, res.resname, lig_atom.name, res_atom.name))

    rows = sorted(rows, key=lambda x: x[0])

    print("=== Frame 0: LIG carbon に近い残基 top 30 ===")
    for d, resid, resname, lig_atom, res_atom in rows[:30]:
        print(f"{d:6.2f} Å  {resname:>3} {resid:>4}   LIG {lig_atom} - {resname} {res_atom}")



# ============================================================
# 16. RMSD
# ============================================================

if RUN_RMSD:
    print('\n=== 16. RMSD ===')
    # --- Original notebook cell 40 ---
    #chatgpt修正版

    #@title **Compute RMSD of protein's CA atoms**
    #@markdown **Provide output file names below:**
    Output_name = 'rmsd_ca' #@param {type:"string"}


    rmsd = pt.rmsd(traj_load, ref = 0, mask = "@CA")

    time = len(rmsd)*int(Write_the_trajectory)/1000
    time_array = np.arange(0,time,int(Write_the_trajectory)/1000)*int(stride_traj)

    # Plotting:
    ax = plt.plot(time_array, rmsd, alpha=0.6, color = 'blue', linewidth = 1.0)
    plt.xlim(0, simulation_ns)
    #plt.ylim(2, 6)

    plt.xlabel("Time (ns)", fontsize = 14, fontweight = 'bold')
    plt.ylabel("RMSD [$\AA$]", fontsize = 14, fontweight = 'bold')
    plt.xticks(fontsize = 12)
    plt.yticks(fontsize = 12)
    plt.savefig(os.path.join(workDir, Output_name + ".png"), dpi=600, bbox_inches='tight')

    raw_data=pd.DataFrame(rmsd)
    raw_data.to_csv(os.path.join(workDir, Output_name + ".csv"))



# ============================================================
# 17. RMSD distribution
# ============================================================

if RUN_RMSD_DISTRIBUTION:
    print('\n=== 17. RMSD distribution ===')
    # --- Original notebook cell 41 ---
    #chatgpt修正版

    #@title **Plot RMSD as a ditribution**

    #@markdown **Provide output file names below:**
    Output_name = 'rmsd_dist' #@param {type:"string"}

    ax = sb.kdeplot(rmsd, color="blue", shade=True, alpha=0.2, linewidth=0.5)
    plt.xlabel('RMSD [$\AA$]', fontsize = 14, fontweight = 'bold')
    plt.xticks(fontsize = 12)
    plt.yticks([])
    plt.ylabel('')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(False)

    plt.savefig(os.path.join(workDir, Output_name + ".png"), dpi=600, bbox_inches='tight')



# ============================================================
# 18. Radius of gyration
# ============================================================

if RUN_RADIUS_GYRATION:
    print('\n=== 18. Radius of gyration ===')
    # --- Original notebook cell 42 ---
    #chatgpt修正版

    #@title **Compute radius of gyration of protein's CA atoms**

    #@markdown **Provide output file names below:**
    Output_name = 'radius_gyration' #@param {type:"string"}

    radgyr = pt.radgyr(traj_load, mask = "@CA")

    time = len(radgyr)*int(Write_the_trajectory)/1000
    time_array = np.arange(0,time,int(Write_the_trajectory)/1000)*int(stride_traj)

    # Plotting:
    plt.plot(time_array, radgyr, alpha=0.6, color='green', linewidth=1.0)
    plt.xlim(0, simulation_ns)

    plt.xlabel("Time (ns)", fontsize=14, fontweight='bold')
    plt.ylabel("Radius of gyration ($\AA$)", fontsize=14, fontweight='bold')
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    plt.savefig(os.path.join(workDir, Output_name + ".png"), dpi=600, bbox_inches='tight')
    plt.show()

    raw_data = pd.DataFrame(radgyr)
    raw_data.to_csv(os.path.join(workDir, Output_name + ".csv"))



# ============================================================
# 19. Radius of gyration distribution
# ============================================================

if RUN_RADIUS_GYRATION_DISTRIBUTION:
    print('\n=== 19. Radius of gyration distribution ===')
    # --- Original notebook cell 43 ---
    #chatgpt 修正版

    #@title **Plot radius of gyration as a ditribution**

    #@markdown **Provide output file names below:**
    Output_name = 'radius_gyration_dist' #@param {type:"string"}

    plt.figure()
    ax = sb.kdeplot(radgyr, color="green", fill=True, alpha=0.2, linewidth=0.5)
    plt.xlabel('Radius of gyration ($\AA$)', fontsize = 14, fontweight = 'bold')
    plt.xticks(fontsize = 12)
    plt.yticks([])
    plt.ylabel('')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(False)

    plt.savefig(os.path.join(workDir, Output_name + ".png"), dpi=600, bbox_inches='tight')
    plt.show()



# ============================================================
# 20. RMSF
# ============================================================

if RUN_RMSF:
    print('\n=== 20. RMSF ===')
    # --- Original notebook cell 44 ---
    #chatgpt修正版

    #@title **Compute RMSF of protein's CA atoms**

    #@markdown **Provide output file names below:**
    Output_name = 'rmsf_ca' #@param {type:"string"}

    rmsf = pt.rmsf(traj_load, "@CA")
    bfactor = pt.bfactors(traj_load, byres=True)

    # CA原子に対応する残基番号を取得
    ca_indices = traj_load.top.select("@CA")
    residue_numbers = []

    for atom_index in ca_indices:
        atom = traj_load.top.atom(atom_index)
        residue_numbers.append(atom.resid + 1)

    # Plotting:
    plt.plot(residue_numbers, rmsf[:,1], alpha=1.0, color='red', linewidth=1.0)

    plt.xlabel("Residue", fontsize=14, fontweight='bold')
    plt.ylabel("RMSF ($\\AA$)", fontsize=14, fontweight='bold')
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.xlim(min(residue_numbers), max(residue_numbers))

    plt.savefig(os.path.join(workDir, Output_name + ".png"), dpi=600, bbox_inches='tight')
    plt.show()

    raw_data = pd.DataFrame({
        "Residue": residue_numbers,
        "RMSF": rmsf[:,1]
    })
    raw_data.to_csv(os.path.join(workDir, Output_name + ".csv"), index=False)



# ============================================================
# 21. 2D RMSD
# ============================================================

if RUN_2D_RMSD:
    print('\n=== 21. 2D RMSD ===')
    # --- Original notebook cell 45 ---
    #chatgpt修正版

    #@title **2D RMSD**

    #@markdown **Provide output file names below:**
    Output_name = '2D_rmsd' #@param {type:"string"}

    plt.figure()

    last_frame = len(time_array)

    stride_ticks_f = last_frame / 5
    ticks_frame = np.arange(0, (len(time_array) + float(stride_ticks_f)), float(stride_ticks_f))
    a = ticks_frame.astype(float)

    stride_ticks_t = simulation_ns / 5
    tick_time = np.arange(0, (float(simulation_ns) + float(stride_ticks_t)), float(stride_ticks_t))
    b = tick_time.astype(float)

    mat1 = pt.pairwise_rmsd(traj_load, mask="@CA", frame_indices=range(len(traj_load)))

    ax = plt.imshow(mat1, cmap='PRGn', origin='lower', interpolation='bicubic')
    plt.title('2D RMSD')
    plt.xlabel('Time (ns)', fontsize=14, fontweight='bold')
    plt.ylabel('Time (ns)', fontsize=14, fontweight='bold')
    plt.xticks(a, b.round(decimals=3), fontsize=12)
    plt.yticks(a, b.round(decimals=3), fontsize=12)

    cbar1 = plt.colorbar()
    cbar1.set_label("RMSD ($\\AA$)", fontsize=14, fontweight='bold')

    plt.savefig(os.path.join(workDir, Output_name + ".png"), dpi=600, bbox_inches='tight')
    plt.show()

    raw_data = pd.DataFrame(mat1)
    raw_data.to_csv(os.path.join(workDir, Output_name + ".csv"), index=False)



# ============================================================
# 22. PCA
# ============================================================

if RUN_PCA:
    print('\n=== 22. PCA ===')
    # --- Original notebook cell 46 ---
    #chatgpt修正版

    #@title **Calculate eigvenctors of Principle Component Analysis (PCA)**
    data = pt.pca(traj_load, fit=True, ref=0, mask='@CA', n_vecs=2)

    #@markdown **Provide output file names below:**
    Output_name = 'PCA' #@param {type:"string"}

    Output_PC1 = 'PC1' #@param {type:"string"}
    Output_PC2 = 'PC2' #@param {type:"string"}

    projection_data = data[0]
    PC1 = data[0][0]
    PC2 = data[0][1]

    nframes = len(PC1)
    last_frame = nframes

    stride_ticks_f = last_frame / 5
    ticks_frame = np.arange(0, last_frame + stride_ticks_f, stride_ticks_f)
    tick_positions = ticks_frame.astype(float)
    tick_positions_list = tick_positions.tolist()

    stride_ticks_t = simulation_ns / 5
    tick_time = np.arange(0, float(simulation_ns) + float(stride_ticks_t), float(stride_ticks_t))
    tick_labels = tick_time.astype(float)

    plt.figure()
    plt.title(r'PCA of C-$\alpha$')

    sc = plt.scatter(PC1, PC2, c=range(nframes), cmap='Greens', marker='o', s=8, alpha=1)
    plt.clim(0, last_frame)

    plt.xlabel('PC1', fontsize=14, fontweight='bold')
    plt.ylabel('PC2', fontsize=14, fontweight='bold')
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    cbar1 = plt.colorbar(sc, orientation="vertical")
    cbar1.set_label('Time(ns)', fontsize=14, fontweight='bold')
    cbar1.set_ticks(tick_positions_list)
    cbar1.set_ticklabels(tick_labels.round(decimals=3))

    plt.savefig(os.path.join(workDir, Output_name + ".png"), dpi=600, bbox_inches='tight')
    plt.show()

    pc1 = pd.DataFrame(PC1)
    pc1.to_csv(os.path.join(workDir, Output_PC1 + ".csv"), index=False)

    pc2 = pd.DataFrame(PC2)
    pc2.to_csv(os.path.join(workDir, Output_PC2 + ".csv"), index=False)



# ============================================================
# 23. PCA distribution
# ============================================================

if RUN_PCA_DISTRIBUTION:
    print('\n=== 23. PCA distribution ===')
    # --- Original notebook cell 47 ---
    #chatgpt修正版

    #@title **Plot Principal Component 1 (PC1) and Principal Component 2 (PC2) as a ditribution**
    Output_name = 'PCA_dist' #@param {type:"string"}

    fig = plt.figure(figsize=(9,5))

    plt.subplot(1, 2, 1)
    ax = sb.kdeplot(PC1, color="green", fill=True, alpha=0.2, linewidth=0.5)
    plt.xlabel('PC1', fontsize = 14, fontweight = 'bold')
    plt.xticks(fontsize = 12)
    plt.yticks([])
    plt.ylabel('')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(False)

    plt.subplot(1, 2, 2)
    ax2 = sb.kdeplot(PC2, color="purple", fill=True, alpha=0.2, linewidth=0.5)
    plt.xlabel('PC2', fontsize = 14, fontweight = 'bold')
    plt.xticks(fontsize = 12)
    plt.yticks([])
    plt.ylabel('')
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.spines['bottom'].set_visible(True)
    ax2.spines['left'].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(workDir, Output_name + ".png"), dpi=600, bbox_inches='tight')
    plt.show()



# ============================================================
# 24. Pearson Cross Correlation
# ============================================================

if RUN_CROSS_CORRELATION:
    print('\n=== 24. Pearson Cross Correlation ===')
    # --- Original notebook cell 48 ---
    #chatgpt修正版

    #@title **Pearson's Cross Correlation (CC)**

    #@markdown **Provide output file names below:**
    Output_name = 'cross_correlation' #@param {type:"string"}

    plt.figure()

    traj_align = pt.align(traj_load, mask='@CA', ref=0)
    mat_cc = matrix.correl(traj_align, '@CA')

    ax = plt.imshow(mat_cc, cmap='PiYG_r', interpolation='bicubic', vmin=-1, vmax=1, origin='lower')

    plt.xlabel('Residues', fontsize=14, fontweight='bold')
    plt.ylabel('Residues', fontsize=14, fontweight='bold')
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    cbar1 = plt.colorbar()
    cbar1.set_label('$CC_{ij}$', fontsize=14, fontweight='bold')

    plt.savefig(os.path.join(workDir, Output_name + ".png"), dpi=600, bbox_inches='tight')
    plt.show()

    raw_data = pd.DataFrame(mat_cc)
    raw_data.to_csv(os.path.join(workDir, Output_name + ".csv"), index=False)



# ============================================================
# 25. Contact frequency for selected residue range
# ============================================================

if RUN_CONTACT_FREQUENCY_RANGE:
    print('\n=== 25. Contact frequency for selected residue range ===')
    # --- Original notebook cell 49 ---
    #chatgpt修正版

    #@title **Ligand-Protein Contact Analysis for Selected Residue Range**
    # 指定した残基範囲について、各残基とリガンドの最短距離・接触頻度を計算する

    import os
    import MDAnalysis as mda
    from MDAnalysis.lib.distances import distance_array
    import numpy as np
    import matplotlib.pyplot as plt
    import pandas as pd

    # ============================================================
    # 入力設定
    # ============================================================

    # 今のGPUサーバー側の作業ディレクトリ
    workDir = "/home/gregorymaddux18/making_it_rain_work"

    # trajectory連結セルで作ったファイルを使う
    topology_path = os.path.join(workDir, "SYS_gaff2.prmtop")
    trajectory_path = os.path.join(workDir, "prot_lig_prod1-30_whole.dcd")

    # 解析したい残基範囲
    start_res = 100
    end_res = 200

    # 接触判定距離 cutoff Å
    cutoff = 5.0

    # 出力ファイル名
    csv_filename = f"residue_distances_{start_res}-{end_res}.csv"
    summary_csv_filename = f"residue_contact_frequency_{start_res}-{end_res}.csv"
    png_filename = f"residue_contact_frequency_{start_res}-{end_res}.png"

    csv_path = os.path.join(workDir, csv_filename)
    summary_csv_path = os.path.join(workDir, summary_csv_filename)
    png_path = os.path.join(workDir, png_filename)

    # ============================================================
    # ファイル確認
    # ============================================================

    if not os.path.exists(topology_path):
        raise FileNotFoundError(f"Topology file not found: {topology_path}")

    if not os.path.exists(trajectory_path):
        raise FileNotFoundError(f"Trajectory file not found: {trajectory_path}")

    print("Topology:", topology_path)
    print("Trajectory:", trajectory_path)

    # ============================================================
    # Universe読み込み
    # ============================================================

    u = mda.Universe(topology_path, trajectory_path)

    # 対象選択
    protein_range = u.select_atoms(f"protein and resid {start_res}:{end_res}")
    ligand = u.select_atoms("resname LIG")

    if len(protein_range) == 0:
        raise ValueError(f"resid {start_res}-{end_res} のタンパク質原子が見つかりません。残基番号を確認してください。")

    if len(ligand) == 0:
        raise ValueError("resname LIG のリガンドが見つかりません。")

    residues_in_range = protein_range.residues
    n_frames = len(u.trajectory)

    print(f"解析中... 対象 resid {start_res}-{end_res}")
    print("Frames:", n_frames)
    print("Ligand atoms:", len(ligand))
    print("Target residues:", len(residues_in_range))

    # ============================================================
    # 距離計算
    # 各フレームごとに、LIG全原子 vs 各残基の重原子の最短距離を計算
    # ============================================================

    distance_data = {"Frame": np.arange(1, n_frames + 1)}
    contact_results = []

    for res in residues_in_range:
        label = f"{res.resname}{res.resid}"

        # 水素を除いた残基重原子
        res_atoms = res.atoms.select_atoms("not name H*")

        if len(res_atoms) == 0:
            print(f"Warning: {label} に重原子がありません。スキップします。")
            continue

        res_distances = []

        for ts in u.trajectory:
            # PBC情報がある場合はboxを考慮
            box = u.dimensions
            if box is None or np.any(box[:3] == 0):
                box = None

            dmat = distance_array(ligand.positions, res_atoms.positions, box=box)
            min_dist = np.min(dmat)
            res_distances.append(min_dist)

        res_distances = np.array(res_distances)
        contact_frequency = np.mean(res_distances <= cutoff)

        distance_data[label] = res_distances
        contact_results.append({
            "Residue": label,
            "Residue_ID": res.resid,
            "Residue_Name": res.resname,
            "Contact_Frequency": contact_frequency,
            "Mean_Min_Distance_A": np.mean(res_distances),
            "Std_Min_Distance_A": np.std(res_distances),
            "Min_Distance_A": np.min(res_distances),
            "Max_Distance_A": np.max(res_distances),
        })

    # ============================================================
    # CSV保存
    # ============================================================

    df_dist = pd.DataFrame(distance_data)
    df_dist.to_csv(csv_path, index=False)

    summary_df = pd.DataFrame(contact_results)
    summary_df.to_csv(summary_csv_path, index=False)

    print(f"成功: 距離データを保存しました: {csv_path}")
    print(f"成功: 接触頻度サマリーを保存しました: {summary_csv_path}")

    # ============================================================
    # グラフ描画
    # ============================================================

    labels = summary_df["Residue"].tolist()
    freqs = summary_df["Contact_Frequency"].tolist()

    plt.figure(figsize=(10, max(6, len(labels) * 0.25)))

    colors = ["teal" if f > 0 else "lightgrey" for f in freqs]
    bars = plt.barh(labels, freqs, color=colors, edgecolor="black", alpha=0.8)

    plt.xlabel("Contact Frequency (0.0 - 1.0)", fontsize=12)
    plt.ylabel(f"Amino Acid Residue ({start_res}-{end_res})", fontsize=12)
    plt.title(f"Ligand-Protein Contact Analysis (resid {start_res}-{end_res}, cutoff {cutoff} Å)", fontsize=14)
    plt.xlim(0, 1.1)
    plt.grid(axis="x", linestyle="--", alpha=0.5)

    for bar in bars:
        width = bar.get_width()
        if width > 0:
            plt.text(
                width + 0.01,
                bar.get_y() + bar.get_height() / 2,
                f"{width:.2f}",
                va="center",
                fontsize=9,
                fontweight="bold"
            )

    plt.tight_layout()
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"成功: グラフを保存しました: {png_path}")



print('\n=== Integrated pipeline cell finished ===')
